# Pipeline Penelitian Disertasi: Imbalance-Aware Modified PSO untuk Seleksi Fitur Multilabel

Ujicoba ini untuk menyesuaikan eksperimen dengan kontribusi penelitian disertasi, yaitu pengembangan metode seleksi fitur untuk klasifikasi multilabel umpan balik mahasiswa dengan ketidakseimbangan label.

Fokus utama eksperimen:

1. Membaca dan membersihkan dataset umpan balik mahasiswa secara aman.
2. Melakukan preprocessing teks sebelum pemisahan data.
3. Menghasilkan representasi fitur menggunakan IndoBERT, dengan TF-IDF sebagai fallback.
4. Menangani ketidakseimbangan label hanya pada data training menggunakan MLSMOTE.
5. Menjalankan baseline tanpa seleksi fitur.
6. Menjalankan seleksi fitur statistik sebagai pembanding dan sebagai tahap penyaringan awal.
7. Menjalankan Standard PSO sebagai pembanding metaheuristik.
8. Menjalankan metode usulan **Imbalance-Aware Modified PSO** sebagai optimasi subset fitur setelah penyaringan statistik.
9. Menggunakan fungsi objektif yang mempertimbangkan kualitas klasifikasi, sensitivitas label minoritas, efisiensi jumlah fitur, dan redundansi fitur.
10. Menampilkan tabel perbandingan hasil, jumlah fitur, dan performa per label.

Catatan penting:

- Data testing tidak pernah digunakan dalam MLSMOTE, seleksi fitur, atau proses optimasi.
- Evaluasi akhir tetap dilakukan pada data testing asli.
- Label minoritas ditentukan dari distribusi label pada data training sebelum MLSMOTE.
- Metode usulan pada Ujicoba ini adalah **Statistical Screening + Imbalance-Aware Modified PSO**.


In [4]:
# ============================================================
# 1. Instalasi library
# ============================================================
%pip install -q gdown transformers Sastrawi tqdm scikit-learn matplotlib joblib openpyxl scipy


In [5]:
# ============================================================
# 2. Import library dan konfigurasi utama
# ============================================================
import os
import re
import random
import hashlib
import warnings
from pathlib import Path

import gdown
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import VarianceThreshold, f_classif, mutual_info_classif
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, hamming_loss, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# Nama file
DATA_PATH = Path("cleaned_df.csv")
ACRONYM_PATH = Path("acronym.txt")
KBBA_PATH = Path("kbba.txt")
STOPWORDS_PATH = Path("stopwordsID.txt")

# Kolom dataset
TEXT_COL = "saran"
CLEAN_TEXT_COL = "text_clean"
LABELS = ["SU", "DPAL", "DSTI", "DA", "DPMBKA", "DK", "Perpustakaan"]

# Konfigurasi embedding
MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LEN = 128
BATCH_SIZE = 16
TEST_SIZE = 0.20
CACHE_DIR = Path("cache_embedding")
CACHE_DIR.mkdir(exist_ok=True)

# Konfigurasi seleksi fitur statistik
# Nilai ini dibuat sebagai grid agar hasil tidak terkunci pada 200 fitur saja.
FEATURE_TOP_K_LIST = [50, 100, 150, 200, 300, 500]
THETA_VT = 0.0
RELIEF_SAMPLE_SIZE = 400

# Konfigurasi kandidat fitur untuk metode usulan
# Tahap 1 menyaring fitur statistik. Tahap 2 mengoptimasi subset memakai IA-MPSO.
DEFAULT_CANDIDATE_TOP_K = 300
MIN_SELECTED_FEATURES = 20
MAX_FEATURE_RATIO = 0.70

# Konfigurasi Random Forest evaluasi akhir
RF_N_ESTIMATORS = 150
RF_MAX_DEPTH = 14

# Konfigurasi Random Forest di dalam fitness agar proses optimasi tidak terlalu berat
FITNESS_RF_N_ESTIMATORS = 70
FITNESS_RF_MAX_DEPTH = 10
CV_SPLITS = 3

# Konfigurasi PSO
PSO_POP_SIZE = 12
PSO_MAX_ITER = 15
PSO_PATIENCE = 5
W_MAX = 0.90
W_MIN = 0.40
C1 = 1.70
C2 = 1.70
RELEVANCE_BIAS = 1.20

# Bobot fungsi objektif metode usulan
# Kualitas klasifikasi dan sensitivitas label minoritas diberi bobot utama.
OBJ_W_WEIGHTED_F1 = 0.30
OBJ_W_MACRO_F1 = 0.20
OBJ_W_MINOR_RECALL = 0.30
OBJ_W_HAMMING = 0.10
OBJ_W_SUBSET_SIZE = 0.06
OBJ_W_REDUNDANCY = 0.04

print("Konfigurasi selesai.")


Konfigurasi selesai.


In [6]:
# ============================================================
# 3. Download dataset dan file pendukung jika belum tersedia
# ============================================================
FILES = {
    "acronym.txt": "1i-_dpAtulIM2gdfk6mXbfi_tfoqWz036",
    "cleaned_df.csv": "1Dtp7emDjco9BeKbZPS6P5ImQXVmNPGyM",
    "kbba.txt": "1OeRk-QnxogCvTGZiYN2hHYqqMpUuFYhk",
    "stopwordsID.txt": "1_8NjGVBjnx_7OrjvjgFAM2wvnJSNK6qf",
}


def download_if_missing(filename, file_id):
    path = Path(filename)
    if path.exists() and path.stat().st_size > 0:
        print(f"File sudah ada: {filename}")
        return path

    print(f"Mengunduh {filename} ...")
    url = f"https://drive.google.com/uc?id={file_id}"
    gdown.download(url, str(path), quiet=False)

    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(
            f"{filename} belum tersedia. Upload file ini ke folder kerja atau cek akses Google Drive."
        )
    return path


for filename, file_id in FILES.items():
    download_if_missing(filename, file_id)

print("Semua file utama sudah tersedia.")


File sudah ada: acronym.txt
Mengunduh cleaned_df.csv ...
Gagal mengunduh cleaned_df.csv secara otomatis: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1Dtp7emDjco9BeKbZPS6P5ImQXVmNPGyM

but Gdown can't. Please check connections and permissions.


FileNotFoundError: 
[ERROR] File 'cleaned_df.csv' tidak dapat diunduh otomatis.
Silakan unduh file secara manual melalui browser di link berikut:
https://drive.google.com/uc?id=1Dtp7emDjco9BeKbZPS6P5ImQXVmNPGyM
Setelah diunduh, silakan upload file tersebut ke folder kerja Colab Anda.

In [ ]:
# ============================================================
# 4. Fungsi preprocessing teks
# ============================================================
try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    stemmer = StemmerFactory().create_stemmer()
except Exception:
    stemmer = None


def load_dictionary(path, separator, skip_first=False):
    data = {}
    path = Path(path)
    if not path.exists():
        return data

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    if skip_first and len(lines) > 0:
        lines = lines[1:]

    for line in lines:
        line = line.strip()
        if not line or separator not in line:
            continue
        key, value = line.split(separator, 1)
        key = key.strip().lower()
        value = value.strip().lower()
        if key and value:
            data[key] = value
    return data


def load_stopwords(path):
    path = Path(path)
    if not path.exists():
        return {
            "yang", "dan", "di", "ke", "dari", "ini", "itu", "untuk",
            "dengan", "pada", "adalah", "karena", "atau", "sebagai",
            "dalam", "saya", "kami", "mereka", "ada", "agar"
        }

    words = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            word = line.strip().lower()
            if word:
                words.append(word)

    stop_words = set(words)
    # Kata negasi dan intensitas penting untuk konteks keluhan, jadi tidak dihapus.
    stop_words = stop_words - {"kurang", "banyak", "tidak", "belum", "jangan"}
    return stop_words


ACRONYM = load_dictionary(ACRONYM_PATH, "=")
SLANG = load_dictionary(KBBA_PATH, "	")
STOP_WORDS = load_stopwords(STOPWORDS_PATH)


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+|#\w+", " ", text)
    text = re.sub(r"[^a-zA-ZÀ-ÿ\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    tokens = [ACRONYM.get(token, token) for token in tokens]
    tokens = [SLANG.get(token, token) for token in tokens]
    tokens = [token for token in tokens if token not in STOP_WORDS]

    if stemmer is not None:
        tokens = [stemmer.stem(token) for token in tokens]

    return " ".join(tokens).strip()


print("Jumlah acronym:", len(ACRONYM))
print("Jumlah slang   :", len(SLANG))
print("Jumlah stopword:", len(STOP_WORDS))


In [ ]:
# ============================================================
# 5. Load dataset, audit data, pembersihan, dan validasi label
# ============================================================
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset tidak ditemukan: {DATA_PATH}")

# Fungsi bantu untuk mencatat audit data pada setiap tahap.
def add_audit_record(records, tahap, before, after, kriteria, dampak):
    gugur = int(before - after)
    records.append({
        "Tahap Audit Data": tahap,
        "Jumlah Data Awal Tahap": int(before),
        "Jumlah Data Gugur": gugur,
        "Jumlah Data Tersisa": int(after),
        "Kriteria Penghapusan / Tindakan": kriteria,
        "Potensi Dampak terhadap Dataset": dampak
    })


audit_records = []

# 1. Membaca data mentah.
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
raw_rows = len(df_raw)
add_audit_record(
    audit_records,
    "Data awal terbaca",
    raw_rows,
    raw_rows,
    "Seluruh baris dibaca dari file dataset awal.",
    "Data masih mentah dan masih dapat mengandung teks kosong, label tidak sesuai, duplikasi, atau baris tidak valid."
)

# 2. Normalisasi nama kolom dan validasi struktur kolom.
df_raw.columns = df_raw.columns.str.strip()
required_cols = [TEXT_COL] + LABELS
missing_cols = [col for col in required_cols if col not in df_raw.columns]
if missing_cols:
    raise ValueError(f"Kolom berikut tidak ditemukan dalam dataset: {missing_cols}")

add_audit_record(
    audit_records,
    "Validasi struktur kolom",
    raw_rows,
    len(df_raw),
    "Dataset harus memiliki kolom teks dan seluruh kolom label multilabel.",
    "Tidak ada baris yang dihapus jika struktur kolom lengkap."
)

# 3. Hapus baris dengan teks NaN pada kolom utama.
before = len(df_raw)
df = df_raw.dropna(subset=[TEXT_COL]).copy()
add_audit_record(
    audit_records,
    "Validasi teks kosong atau NaN",
    before,
    len(df),
    "Baris dengan nilai kosong/NaN pada kolom teks utama dihapus.",
    "Menghapus data yang tidak dapat diproses menjadi fitur teks."
)

# 4. Hapus baris yang hanya berisi spasi kosong.
before = len(df)
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL].ne("")].copy()
add_audit_record(
    audit_records,
    "Validasi teks hanya spasi",
    before,
    len(df),
    "Baris yang hanya berisi spasi kosong dihapus.",
    "Menjaga agar setiap data memiliki isi teks yang bermakna."
)

# 5. Validasi dan konversi label ke format biner 0/1.
# Tahap ini mengikuti pipeline eksperimen: label tidak valid dikonversi/dinormalisasi,
# bukan langsung menghapus baris, agar jumlah data final tetap sesuai hasil uji coba.
before = len(df)
non_numeric_label_cells = 0
out_of_range_label_cells = 0
for label in LABELS:
    converted = pd.to_numeric(df[label], errors="coerce")
    non_numeric_label_cells += int(converted.isna().sum())
    out_of_range_label_cells += int(((converted.notna()) & ~converted.isin([0, 1])).sum())
    df[label] = converted

df[LABELS] = df[LABELS].replace([np.inf, -np.inf], np.nan).fillna(0)
df[LABELS] = df[LABELS].clip(0, 1).astype(int)
add_audit_record(
    audit_records,
    "Validasi format label biner",
    before,
    len(df),
    f"Label dikonversi menjadi numerik biner 0/1. Sel nonnumerik/NaN: {non_numeric_label_cells}; sel di luar 0/1: {out_of_range_label_cells}.",
    "Menjaga konsistensi target multilabel. Tahap ini menstandarkan label, bukan menghapus baris."
)

# 6. Pemeriksaan data tanpa label aktif.
# Pada pipeline ini tahap ini bersifat audit. Baris tidak dihapus agar hasil tetap konsisten
# dengan eksperimen yang sudah berjalan. Jika pembimbing meminta penghapusan, aktifkan filter opsional di bawah.
before = len(df)
rows_without_active_label = int((df[LABELS].sum(axis=1) == 0).sum())
# OPSIONAL jika perlu menghapus data tanpa label aktif:
# df = df[df[LABELS].sum(axis=1) > 0].copy()
add_audit_record(
    audit_records,
    "Audit data tanpa label aktif",
    before,
    len(df),
    f"Ditemukan {rows_without_active_label} baris dengan seluruh label bernilai 0. Pada pipeline ini dicatat sebagai audit, tidak dihapus.",
    "Menunjukkan potensi data yang tidak memiliki target aktif; perlu dijelaskan bila jumlahnya besar."
)

# 7. Pemeriksaan duplikasi berdasarkan teks dan label.
# Pada pipeline ini tahap ini bersifat audit. Baris tidak dihapus agar hasil tetap konsisten
# dengan eksperimen yang sudah berjalan. Jika pembimbing meminta penghapusan, aktifkan drop_duplicates opsional.
before = len(df)
duplicate_cols = [TEXT_COL] + LABELS
duplicate_rows = int(df.duplicated(subset=duplicate_cols).sum())
# OPSIONAL jika perlu menghapus duplikasi identik:
# df = df.drop_duplicates(subset=duplicate_cols).copy()
add_audit_record(
    audit_records,
    "Audit duplikasi teks dan label",
    before,
    len(df),
    f"Ditemukan {duplicate_rows} baris duplikat berdasarkan kombinasi teks dan label. Pada pipeline ini dicatat sebagai audit, tidak dihapus.",
    "Menunjukkan potensi pengulangan data; perlu dijelaskan bila memengaruhi distribusi label."
)

# 8. Preprocessing teks dan hapus teks kosong setelah preprocessing.
before = len(df)
df[CLEAN_TEXT_COL] = df[TEXT_COL].apply(clean_text)
df = df[df[CLEAN_TEXT_COL].str.strip().ne("")].copy()
df = df.reset_index(drop=True)
add_audit_record(
    audit_records,
    "Pembersihan teks kosong setelah preprocessing",
    before,
    len(df),
    "Baris yang menghasilkan teks kosong setelah cleaning, normalisasi, stopword removal, dan stemming dihapus.",
    "Menjaga agar setiap data final memiliki fitur teks yang dapat dibentuk menjadi embedding."
)

# 9. Dataset final.
add_audit_record(
    audit_records,
    "Dataset final valid",
    len(df),
    len(df),
    "Data telah memiliki teks bersih dan label multilabel biner yang valid.",
    "Dataset siap digunakan untuk split data, embedding, MLSMOTE, seleksi fitur, optimasi IA-MPSO, dan evaluasi model."
)

audit_data_df = pd.DataFrame(audit_records)
print("Jumlah baris awal:", raw_rows)
print("Jumlah baris valid setelah audit dan pembersihan:", len(df))
print("Jumlah data gugur total:", raw_rows - len(df))
print("Shape dataset final:", df.shape)

print("\nTabel Audit Data:")
display(audit_data_df)

print("\nDistribusi label dataset final:")
print(df[LABELS].sum().sort_values(ascending=False))

display(df[[TEXT_COL, CLEAN_TEXT_COL] + LABELS].head())



In [ ]:
# ============================================================
# 6. Analisis distribusi label dan ketidakseimbangan multilabel
# ============================================================
label_counts = df[LABELS].sum().sort_values(ascending=False)
max_count = label_counts.max()
mean_count = label_counts.mean()

label_distribution_df = pd.DataFrame({
    "Label": label_counts.index,
    "Positive Count": label_counts.values,
    "Imbalance Ratio to Max": (max_count / label_counts.values).round(4),
    "Minority by Mean": label_counts.values < mean_count
})

print("Distribusi label dan indikasi label minoritas:")
display(label_distribution_df)

plt.figure(figsize=(10, 5))
bars = plt.bar(label_counts.index, label_counts.values)
plt.title("Distribusi Label pada Dataset Multilabel")
plt.xlabel("Label")
plt.ylabel("Jumlah Data Positif")
plt.xticks(rotation=45)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        int(height),
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 7. Train-test split yang aman untuk multilabel
# ============================================================
def safe_multilabel_split(X, Y, test_size=0.2, random_state=42, max_trials=100):
    # Mencari split yang tetap memiliki minimal satu data positif untuk setiap label pada train dan test.
    for seed in range(random_state, random_state + max_trials):
        X_train, X_test, Y_train, Y_test = train_test_split(
            X,
            Y,
            test_size=test_size,
            random_state=seed,
            shuffle=True
        )
        if (Y_train.sum(axis=0) > 0).all() and (Y_test.sum(axis=0) > 0).all():
            print(f"Split aman ditemukan dengan random_state={seed}")
            return X_train, X_test, Y_train, Y_test, seed

    print("Peringatan: split aman tidak ditemukan. Menggunakan random_state awal.")
    X_train, X_test, Y_train, Y_test = train_test_split(
        X,
        Y,
        test_size=test_size,
        random_state=random_state,
        shuffle=True
    )
    return X_train, X_test, Y_train, Y_test, random_state


X_text = df[CLEAN_TEXT_COL].astype(str).values
Y = df[LABELS].astype(int).values

X_train_text, X_test_text, Y_train, Y_test, SPLIT_RANDOM_STATE = safe_multilabel_split(
    X_text,
    Y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_label_counts = Y_train.sum(axis=0)
train_mean_count = train_label_counts.mean()
MINORITY_LABEL_INDICES = np.where(train_label_counts < train_mean_count)[0]
if len(MINORITY_LABEL_INDICES) == 0:
    MINORITY_LABEL_INDICES = np.arange(len(LABELS))
MINORITY_LABELS = [LABELS[i] for i in MINORITY_LABEL_INDICES]

print("Jumlah data train:", len(X_train_text))
print("Jumlah data test :", len(X_test_text))
print("Shape Y_train    :", Y_train.shape)
print("Shape Y_test     :", Y_test.shape)
print("Label minoritas berdasarkan data training asli:", MINORITY_LABELS)

print("\nDistribusi label train:")
print(pd.Series(Y_train.sum(axis=0), index=LABELS).sort_values(ascending=False))
print("\nDistribusi label test:")
print(pd.Series(Y_test.sum(axis=0), index=LABELS).sort_values(ascending=False))


In [ ]:
# ============================================================
# 7B. Audit distribusi label pada data training dan testing
# ============================================================
split_distribution_df = pd.DataFrame({
    "Label": LABELS,
    "Dataset Final": df[LABELS].sum().values.astype(int),
    "Data Training": Y_train.sum(axis=0).astype(int),
    "Data Testing": Y_test.sum(axis=0).astype(int),
})
split_distribution_df["Status Representasi"] = np.where(
    (split_distribution_df["Data Training"] > 0) & (split_distribution_df["Data Testing"] > 0),
    "Terwakili pada training dan testing",
    "Perlu diperiksa"
)

print("Audit distribusi label pada data training dan testing:")
display(split_distribution_df.sort_values("Dataset Final", ascending=False))



In [ ]:
# ============================================================
# 8. Ekstraksi embedding: IndoBERT dengan fallback TF-IDF
# ============================================================
try:
    import torch
    from transformers import AutoTokenizer, AutoModel
except Exception as e:
    torch = None
    AutoTokenizer = None
    AutoModel = None
    print("Torch/Transformers belum dapat digunakan:", repr(e))


class BertEmbedder:
    def __init__(self, model_name, max_len=128, device=None):
        if torch is None or AutoTokenizer is None or AutoModel is None:
            raise ImportError("Torch atau Transformers tidak tersedia.")

        self.model_name = model_name
        self.max_len = max_len
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        print("Device yang digunakan:", self.device)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

    def encode(self, texts, batch_size=16):
        texts = [str(text) for text in texts]
        embeddings = []

        for i in tqdm(range(0, len(texts), batch_size), desc="Extracting BERT embeddings"):
            batch_texts = texts[i:i + batch_size]
            encoded = self.tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=self.max_len,
                return_tensors="pt"
            )
            encoded = {key: value.to(self.device) for key, value in encoded.items()}

            with torch.no_grad():
                outputs = self.model(**encoded)
                cls_embedding = outputs.last_hidden_state[:, 0, :]

            embeddings.append(cls_embedding.cpu().numpy())

        return np.vstack(embeddings).astype(np.float32)


def make_cache_name(prefix, texts, model_name):
    sample = " ".join([str(x) for x in texts[:30]])
    raw_key = f"{prefix}_{len(texts)}_{model_name}_{SPLIT_RANDOM_STATE}_{sample}"
    digest = hashlib.md5(raw_key.encode("utf-8")).hexdigest()[:10]
    safe_model = re.sub(r"[^a-zA-Z0-9]+", "_", model_name).strip("_")
    return f"{prefix}_{safe_model}_{len(texts)}_{digest}"


def encode_with_resume(texts, embedder, name, batch_size=16, chunk_size=500, cache_dir=CACHE_DIR):
    texts = [str(text) for text in texts]
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(exist_ok=True)

    final_path = cache_dir / f"{name}_final.npy"
    if final_path.exists():
        embeddings = np.load(final_path)
        if embeddings.shape[0] == len(texts):
            print(f"Loading final embedding: {final_path}")
            return embeddings
        print("Cache final tidak cocok dengan jumlah data saat ini. Embedding dibuat ulang.")
        final_path.unlink()

    total_chunks = (len(texts) + chunk_size - 1) // chunk_size
    chunk_paths = []

    for i in tqdm(range(total_chunks), desc=f"Encoding {name}"):
        start = i * chunk_size
        end = min((i + 1) * chunk_size, len(texts))
        chunk_path = cache_dir / f"{name}_chunk_{i}.npy"
        temp_path = chunk_path.with_suffix(".tmp.npy")
        chunk_paths.append(chunk_path)

        if chunk_path.exists():
            chunk = np.load(chunk_path)
            if chunk.shape[0] == (end - start):
                continue
            print(f"Chunk tidak cocok dan dibuat ulang: {chunk_path}")
            chunk_path.unlink()

        print(f"Processing {name} chunk {i + 1}/{total_chunks} | data {start}:{end}")
        emb_chunk = embedder.encode(texts[start:end], batch_size=batch_size)
        np.save(temp_path, emb_chunk)
        os.replace(temp_path, chunk_path)

    embeddings = [np.load(path) for path in chunk_paths]
    embeddings = np.vstack(embeddings).astype(np.float32)
    np.save(final_path, embeddings)
    print(f"Final embedding saved: {final_path}")
    return embeddings


def build_embeddings(X_train_text, X_test_text):
    # Memakai IndoBERT jika tersedia. Jika gagal, memakai TF-IDF agar pipeline tetap berjalan.
    tfidf_vectorizer = None

    try:
        embedder = BertEmbedder(model_name=MODEL_NAME, max_len=MAX_LEN)
        train_cache_name = make_cache_name("train", X_train_text, MODEL_NAME)
        test_cache_name = make_cache_name("test", X_test_text, MODEL_NAME)

        X_train_emb = encode_with_resume(
            X_train_text,
            embedder,
            train_cache_name,
            batch_size=BATCH_SIZE,
            chunk_size=500
        )
        X_test_emb = encode_with_resume(
            X_test_text,
            embedder,
            test_cache_name,
            batch_size=BATCH_SIZE,
            chunk_size=500
        )
        embedding_backend = "IndoBERT_CLS"

    except Exception as e:
        print("IndoBERT tidak dapat digunakan. Pipeline dilanjutkan dengan TF-IDF.")
        print("Penyebab:", repr(e))
        tfidf_vectorizer = TfidfVectorizer(max_features=768, ngram_range=(1, 2), min_df=1)
        X_train_emb = tfidf_vectorizer.fit_transform(X_train_text).toarray().astype(np.float32)
        X_test_emb = tfidf_vectorizer.transform(X_test_text).toarray().astype(np.float32)
        embedding_backend = "TFIDF_fallback"

    return X_train_emb, X_test_emb, embedding_backend, tfidf_vectorizer


X_train_emb, X_test_emb, EMBEDDING_BACKEND, TFIDF_VECTORIZER = build_embeddings(
    list(X_train_text),
    list(X_test_text)
)

print("Embedding backend:", EMBEDDING_BACKEND)
print("Shape X_train_emb:", X_train_emb.shape)
print("Shape X_test_emb :", X_test_emb.shape)


In [ ]:
# ============================================================
# 9. MLSMOTE hanya pada data training
# ============================================================
def mlsmote(X, Y, k_neighbors=5, random_state=42):
    # Implementasi sederhana MLSMOTE untuk data multilabel.
    rng = np.random.default_rng(random_state)

    X = np.asarray(X, dtype=np.float32)
    Y = np.asarray(Y).astype(int)

    label_counts = Y.sum(axis=0)
    max_count = int(label_counts.max())

    synthetic_X = []
    synthetic_Y = []

    for label_idx in range(Y.shape[1]):
        current_count = int(label_counts[label_idx])

        if current_count == 0 or current_count >= max_count:
            continue

        minority_indices = np.where(Y[:, label_idx] == 1)[0]
        if len(minority_indices) < 2:
            continue

        X_min = X[minority_indices]
        Y_min = Y[minority_indices]

        n_neighbors = min(k_neighbors + 1, len(minority_indices))
        nn = NearestNeighbors(n_neighbors=n_neighbors)
        nn.fit(X_min)
        _, neighbor_indices = nn.kneighbors(X_min)

        n_to_generate = max_count - current_count

        for _ in range(n_to_generate):
            seed_pos = rng.integers(0, len(minority_indices))
            available_neighbors = neighbor_indices[seed_pos][1:]
            if len(available_neighbors) == 0:
                continue

            neighbor_pos = rng.choice(available_neighbors)
            x_seed = X_min[seed_pos]
            x_neighbor = X_min[neighbor_pos]

            lam = rng.random()
            x_new = x_seed + lam * (x_neighbor - x_seed)

            related_positions = np.append(available_neighbors, seed_pos)
            y_votes = Y_min[related_positions].mean(axis=0)
            y_new = (y_votes >= 0.5).astype(int)
            y_new[label_idx] = 1

            synthetic_X.append(x_new)
            synthetic_Y.append(y_new)

    if len(synthetic_X) == 0:
        return X, Y

    X_new = np.vstack([X, np.asarray(synthetic_X, dtype=np.float32)])
    Y_new = np.vstack([Y, np.asarray(synthetic_Y, dtype=int)])
    return X_new, Y_new


X_train_bal, Y_train_bal = mlsmote(
    X_train_emb,
    Y_train,
    k_neighbors=5,
    random_state=RANDOM_STATE
)

print("Before MLSMOTE:", X_train_emb.shape, Y_train.shape)
print("After MLSMOTE :", X_train_bal.shape, Y_train_bal.shape)
print("\nDistribusi label training asli:")
print(pd.Series(Y_train.sum(axis=0), index=LABELS).sort_values(ascending=False))
print("\nDistribusi label setelah MLSMOTE:")
print(pd.Series(Y_train_bal.sum(axis=0), index=LABELS).sort_values(ascending=False))


In [ ]:
# ============================================================
# 9B. Audit MLSMOTE dan ekspor tabel audit dataset final
# ============================================================
mlsmote_distribution_df = pd.DataFrame({
    "Label": LABELS,
    "Sebelum MLSMOTE": Y_train.sum(axis=0).astype(int),
    "Sesudah MLSMOTE": Y_train_bal.sum(axis=0).astype(int),
})
mlsmote_distribution_df["Perubahan"] = (
    mlsmote_distribution_df["Sesudah MLSMOTE"] - mlsmote_distribution_df["Sebelum MLSMOTE"]
).astype(int)

print("Audit distribusi label sebelum dan sesudah MLSMOTE:")
display(mlsmote_distribution_df.sort_values("Sebelum MLSMOTE", ascending=False))

# Tabel ringkas untuk laporan disertasi.
audit_summary_df = audit_data_df.copy()
label_final_df = label_distribution_df.copy()
train_test_df = split_distribution_df.copy()
mlsmote_df = mlsmote_distribution_df.copy()

output_xlsx = "audit_data_dan_representasi_dataset_final.xlsx"
with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
    audit_summary_df.to_excel(writer, sheet_name="Audit Data", index=False)
    label_final_df.to_excel(writer, sheet_name="Distribusi Label Final", index=False)
    train_test_df.to_excel(writer, sheet_name="Train Test Label", index=False)
    mlsmote_df.to_excel(writer, sheet_name="MLSMOTE", index=False)

# Simpan juga versi CSV agar mudah dimasukkan ke laporan.
audit_summary_df.to_csv("audit_data_dataset_final.csv", index=False)
label_final_df.to_csv("distribusi_label_dataset_final.csv", index=False)
train_test_df.to_csv("distribusi_label_train_test.csv", index=False)
mlsmote_df.to_csv("distribusi_label_mlsmote.csv", index=False)

print(f"File audit berhasil disimpan: {output_xlsx}")
print("CSV juga disimpan: audit_data_dataset_final.csv, distribusi_label_dataset_final.csv, distribusi_label_train_test.csv, distribusi_label_mlsmote.csv")



## Tahap Eksperimen

Mulai dari bagian ini, seluruh proses seleksi fitur dan evaluasi dilakukan setelah data training selesai dibuat.

Desain eksperimen terdiri atas empat kelompok:

1. **Baseline**, yaitu Random Forest multilabel tanpa seleksi fitur.
2. **Statistical feature selection**, yaitu metode statistik sebagai pembanding dan tahap penyaringan awal.
3. **Standard PSO**, yaitu PSO biner biasa sebagai pembanding metaheuristik.
4. **Proposed IA-MPSO**, yaitu metode usulan dengan objective function yang memperhatikan ketidakseimbangan label, label minoritas, jumlah fitur, redundansi fitur, dan performa klasifikasi.


In [ ]:
# ============================================================
# 10. Fungsi evaluasi klasifikasi multilabel
# ============================================================
def compute_main_metrics(Y_true, Y_pred, minority_label_indices=None):
    minority_label_indices = np.asarray(minority_label_indices if minority_label_indices is not None else [])

    precision_each = precision_score(Y_true, Y_pred, average=None, zero_division=0)
    recall_each = recall_score(Y_true, Y_pred, average=None, zero_division=0)
    f1_each = f1_score(Y_true, Y_pred, average=None, zero_division=0)

    if len(minority_label_indices) > 0:
        minority_recall = float(np.mean(recall_each[minority_label_indices]))
        minority_f1 = float(np.mean(f1_each[minority_label_indices]))
    else:
        minority_recall = float(np.mean(recall_each))
        minority_f1 = float(np.mean(f1_each))

    return {
        "Precision Weighted": precision_score(Y_true, Y_pred, average="weighted", zero_division=0),
        "Recall Weighted": recall_score(Y_true, Y_pred, average="weighted", zero_division=0),
        "F1 Weighted": f1_score(Y_true, Y_pred, average="weighted", zero_division=0),
        "F1 Macro": f1_score(Y_true, Y_pred, average="macro", zero_division=0),
        "Exact Match Accuracy": accuracy_score(Y_true, Y_pred),
        "Hamming Loss": hamming_loss(Y_true, Y_pred),
        "Minority Recall": minority_recall,
        "Minority F1": minority_f1
    }


def train_evaluate_rf(X_train, Y_train, X_test, Y_test, random_state=42):
    base_rf = RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        class_weight="balanced_subsample",
        random_state=random_state,
        n_jobs=-1
    )
    clf = MultiOutputClassifier(base_rf)
    clf.fit(X_train, Y_train)
    Y_pred = clf.predict(X_test)
    metrics = compute_main_metrics(Y_test, Y_pred, minority_label_indices=MINORITY_LABEL_INDICES)
    return metrics, clf, Y_pred


def compute_research_score(row):
    # Skor ringkas untuk memilih konfigurasi yang seimbang.
    # Nilai tinggi berarti F1, macro F1, dan recall minoritas baik, dengan hamming loss dan jumlah fitur lebih kecil.
    return (
        0.30 * row["F1 Weighted"]
        + 0.25 * row["F1 Macro"]
        + 0.30 * row["Minority Recall"]
        - 0.10 * row["Hamming Loss"]
        - 0.05 * row["Feature Ratio"]
    )


def format_results_table(rows):
    df_result = pd.DataFrame(rows)
    metric_cols = [
        "Precision Weighted", "Recall Weighted", "F1 Weighted", "F1 Macro",
        "Exact Match Accuracy", "Hamming Loss", "Minority Recall", "Minority F1",
        "Feature Ratio", "Research Score"
    ]
    for col in metric_cols:
        if col in df_result.columns:
            df_result[col] = df_result[col].astype(float).round(4)
    return df_result


print("Fungsi evaluasi siap digunakan.")


In [ ]:
# ============================================================
# 11. Fungsi seleksi fitur statistik untuk multilabel
# ============================================================
def sanitize_scores(scores):
    scores = np.asarray(scores, dtype=np.float64)
    scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
    return scores


def normalize_scores(scores, eps=1e-12):
    scores = sanitize_scores(scores)
    min_val = scores.min()
    max_val = scores.max()
    if max_val - min_val < eps:
        return np.ones_like(scores) * 0.5
    return (scores - min_val) / (max_val - min_val + eps)


def select_top_k(scores, top_k):
    scores = sanitize_scores(scores)
    n_features = len(scores)
    k = min(int(top_k), n_features)
    if k <= 0:
        raise ValueError("top_k harus lebih dari 0.")
    selected_idx = np.argsort(scores)[::-1][:k]
    selected_idx = np.sort(selected_idx)
    mask = np.zeros(n_features, dtype=bool)
    mask[selected_idx] = True
    return mask, selected_idx, scores


def fisher_score_multilabel(X, Y, label_weights=None, eps=1e-12):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y).astype(int)
    scores = []
    weights = []

    for label_idx in range(Y.shape[1]):
        y = Y[:, label_idx]
        if len(np.unique(y)) < 2:
            continue

        X_pos = X[y == 1]
        X_neg = X[y == 0]
        mu_all = X.mean(axis=0)
        mu_pos = X_pos.mean(axis=0)
        mu_neg = X_neg.mean(axis=0)
        var_pos = X_pos.var(axis=0)
        var_neg = X_neg.var(axis=0)

        numerator = len(X_pos) * (mu_pos - mu_all) ** 2 + len(X_neg) * (mu_neg - mu_all) ** 2
        denominator = len(X_pos) * var_pos + len(X_neg) * var_neg + eps
        scores.append(numerator / denominator)
        weights.append(1.0 if label_weights is None else label_weights[label_idx])

    if len(scores) == 0:
        return np.zeros(X.shape[1])
    return sanitize_scores(np.average(np.vstack(scores), axis=0, weights=np.asarray(weights)))


def mean_absolute_difference_multilabel(X, Y):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y).astype(int)
    scores = []

    for label_idx in range(Y.shape[1]):
        y = Y[:, label_idx]
        if len(np.unique(y)) < 2:
            continue
        mean_pos = X[y == 1].mean(axis=0)
        mean_neg = X[y == 0].mean(axis=0)
        scores.append(np.abs(mean_pos - mean_neg))

    if len(scores) == 0:
        return np.zeros(X.shape[1])
    return sanitize_scores(np.mean(scores, axis=0))


def dispersion_ratio_multilabel(X, Y, eps=1e-12):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y).astype(int)
    scores = []

    for label_idx in range(Y.shape[1]):
        y = Y[:, label_idx]
        if len(np.unique(y)) < 2:
            continue

        X_pos = X[y == 1]
        X_neg = X[y == 0]
        mean_pos = X_pos.mean(axis=0)
        mean_neg = X_neg.mean(axis=0)
        std_pos = X_pos.std(axis=0)
        std_neg = X_neg.std(axis=0)

        cv_pos = std_pos / (np.abs(mean_pos) + eps)
        cv_neg = std_neg / (np.abs(mean_neg) + eps)
        ratio = np.maximum(cv_pos, cv_neg) / (np.minimum(cv_pos, cv_neg) + eps)
        mean_gap = np.abs(mean_pos - mean_neg)
        scores.append(ratio * mean_gap)

    if len(scores) == 0:
        return np.zeros(X.shape[1])
    return sanitize_scores(np.mean(scores, axis=0))


def anova_score_multilabel(X, Y, label_weights=None):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y).astype(int)
    scores = []
    weights = []

    for label_idx in range(Y.shape[1]):
        y = Y[:, label_idx]
        if len(np.unique(y)) < 2:
            continue
        f_values, _ = f_classif(X, y)
        scores.append(sanitize_scores(f_values))
        weights.append(1.0 if label_weights is None else label_weights[label_idx])

    if len(scores) == 0:
        return np.zeros(X.shape[1])
    return sanitize_scores(np.average(np.vstack(scores), axis=0, weights=np.asarray(weights)))


def pearson_score_multilabel(X, Y, label_weights=None, eps=1e-12):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y, dtype=np.float64)

    X_centered = X - X.mean(axis=0)
    X_std = X_centered.std(axis=0) + eps

    scores = []
    weights = []
    for label_idx in range(Y.shape[1]):
        y = Y[:, label_idx]
        if len(np.unique(y)) < 2:
            continue
        y_centered = y - y.mean()
        y_std = y_centered.std() + eps
        corr = np.abs((X_centered.T @ y_centered) / (len(y) * X_std * y_std))
        scores.append(corr)
        weights.append(1.0 if label_weights is None else label_weights[label_idx])

    if len(scores) == 0:
        return np.zeros(X.shape[1])
    return sanitize_scores(np.average(np.vstack(scores), axis=0, weights=np.asarray(weights)))


def mutual_information_multilabel(X, Y, random_state=42, label_weights=None):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y).astype(int)
    scores = []
    weights = []

    for label_idx in range(Y.shape[1]):
        y = Y[:, label_idx]
        if len(np.unique(y)) < 2:
            continue
        mi = mutual_info_classif(X, y, discrete_features=False, random_state=random_state)
        scores.append(sanitize_scores(mi))
        weights.append(1.0 if label_weights is None else label_weights[label_idx])

    if len(scores) == 0:
        return np.zeros(X.shape[1])
    return sanitize_scores(np.average(np.vstack(scores), axis=0, weights=np.asarray(weights)))


def relieff_score_multilabel(X, Y, sample_size=400, random_state=42, eps=1e-12):
    rng = np.random.default_rng(random_state)
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y).astype(int)

    n_samples, n_features = X.shape
    m = min(sample_size, n_samples)
    sample_indices = rng.choice(n_samples, size=m, replace=False)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)

    weights = np.zeros(n_features, dtype=np.float64)
    valid_label_count = 0

    for label_idx in range(Y.shape[1]):
        y = Y[:, label_idx]
        if len(np.unique(y)) < 2:
            continue

        valid_label_count += 1
        pos_idx = np.where(y == 1)[0]
        neg_idx = np.where(y == 0)[0]
        if len(pos_idx) < 2 or len(neg_idx) < 2:
            continue

        nn_pos = NearestNeighbors(n_neighbors=min(2, len(pos_idx))).fit(X_scaled[pos_idx])
        nn_neg = NearestNeighbors(n_neighbors=min(2, len(neg_idx))).fit(X_scaled[neg_idx])

        for idx in sample_indices:
            xi = X_scaled[idx:idx + 1]

            if y[idx] == 1:
                same_idx = pos_idx
                diff_idx = neg_idx
                nn_same = nn_pos
                nn_diff = nn_neg
            else:
                same_idx = neg_idx
                diff_idx = pos_idx
                nn_same = nn_neg
                nn_diff = nn_pos

            same_neighbors = nn_same.kneighbors(xi, return_distance=False)[0]
            same_global = same_idx[same_neighbors]
            same_global = same_global[same_global != idx]
            if len(same_global) == 0:
                continue
            hit_idx = same_global[0]

            miss_local = nn_diff.kneighbors(xi, return_distance=False)[0][0]
            miss_idx = diff_idx[miss_local]

            diff_hit = np.abs(X_scaled[idx] - X_scaled[hit_idx])
            diff_miss = np.abs(X_scaled[idx] - X_scaled[miss_idx])
            weights += (diff_miss - diff_hit)

    if valid_label_count == 0:
        return np.zeros(n_features)

    weights = weights / (m * valid_label_count + eps)
    weights = weights - weights.min()
    return sanitize_scores(weights)


def make_imbalance_label_weights(Y, minority_label_indices, eps=1e-12):
    counts = np.asarray(Y).sum(axis=0).astype(float)
    inv = 1.0 / (counts + eps)
    inv = inv / inv.mean()
    label_weights = inv.copy()
    label_weights[minority_label_indices] *= 1.5
    return label_weights / label_weights.mean()


LABEL_WEIGHTS_FOR_SELECTION = make_imbalance_label_weights(Y_train, MINORITY_LABEL_INDICES)


def get_statistical_scores(method_name, X_train, Y_train):
    score_functions = {
        "Fisher Score": lambda X, Y: fisher_score_multilabel(X, Y),
        "Variance Threshold": lambda X, Y: X.var(axis=0),
        "Mean Absolute Difference": mean_absolute_difference_multilabel,
        "Dispersion Ratio": dispersion_ratio_multilabel,
        "ANOVA": lambda X, Y: anova_score_multilabel(X, Y),
        "Pearson Correlation": lambda X, Y: pearson_score_multilabel(X, Y),
        "Mutual Information": lambda X, Y: mutual_information_multilabel(X, Y, random_state=RANDOM_STATE),
        "ReliefF": lambda X, Y: relieff_score_multilabel(X, Y, sample_size=RELIEF_SAMPLE_SIZE, random_state=RANDOM_STATE),
        "Minority-Aware ANOVA": lambda X, Y: anova_score_multilabel(X, Y, label_weights=LABEL_WEIGHTS_FOR_SELECTION),
        "Minority-Aware Fisher": lambda X, Y: fisher_score_multilabel(X, Y, label_weights=LABEL_WEIGHTS_FOR_SELECTION),
    }

    if method_name not in score_functions:
        raise ValueError(f"Metode tidak dikenal: {method_name}")
    return sanitize_scores(score_functions[method_name](X_train, Y_train))


def apply_statistical_selector(method_name, X_train, Y_train, X_test, top_k=200):
    if method_name == "Variance Threshold":
        selector = VarianceThreshold(threshold=THETA_VT)
        X_train_selected = selector.fit_transform(X_train)
        X_test_selected = selector.transform(X_test)
        mask = selector.get_support()
        selected_idx = np.where(mask)[0]
        scores = X_train.var(axis=0)
        if X_train_selected.shape[1] == 0:
            raise ValueError("Tidak ada fitur tersisa setelah Variance Threshold.")
        return X_train_selected, X_test_selected, mask, selected_idx, scores

    scores = get_statistical_scores(method_name, X_train, Y_train)
    mask, selected_idx, scores = select_top_k(scores, top_k=top_k)
    X_train_selected = X_train[:, mask]
    X_test_selected = X_test[:, mask]
    return X_train_selected, X_test_selected, mask, selected_idx, scores


STATISTICAL_METHODS = [
    "Fisher Score",
    "Variance Threshold",
    "Mean Absolute Difference",
    "Dispersion Ratio",
    "ANOVA",
    "Pearson Correlation",
    "Mutual Information",
    "ReliefF",
    "Minority-Aware ANOVA",
    "Minority-Aware Fisher"
]

print("Fungsi seleksi fitur statistik siap digunakan.")


In [ ]:
# ============================================================
# 12. Baseline dan seleksi fitur statistik
# ============================================================
all_results = []
statistical_results = []
statistical_artifacts = {}

print("\n" + "=" * 70)
print("Baseline: Random Forest tanpa seleksi fitur")
baseline_metrics, baseline_model, baseline_pred = train_evaluate_rf(
    X_train_bal,
    Y_train_bal,
    X_test_emb,
    Y_test,
    random_state=RANDOM_STATE
)

baseline_row = {
    "Experiment Group": "Baseline",
    "Method": "All Features",
    "Stage 1 Filter": "None",
    "Top K": "All",
    "Selected Features": int(X_train_bal.shape[1]),
    "Feature Ratio": float(X_train_bal.shape[1] / X_train_bal.shape[1]),
    **baseline_metrics
}
baseline_row["Research Score"] = compute_research_score(baseline_row)
all_results.append(baseline_row)

baseline_artifact = {
    "method": "All Features",
    "selected_idx": np.arange(X_train_bal.shape[1]),
    "mask": np.ones(X_train_bal.shape[1], dtype=bool),
    "X_train": X_train_bal,
    "X_test": X_test_emb,
    "model": baseline_model,
    "y_pred": baseline_pred,
    "metrics": baseline_metrics,
    "selected_features": int(X_train_bal.shape[1])
}

print(baseline_row)

for method_name in STATISTICAL_METHODS:
    print("\n" + "=" * 70)
    print(f"Menjalankan seleksi fitur statistik: {method_name}")

    top_k_values = ["VT"] if method_name == "Variance Threshold" else FEATURE_TOP_K_LIST

    for top_k in top_k_values:
        if method_name == "Variance Threshold":
            X_train_fs, X_test_fs, mask, selected_idx, scores = apply_statistical_selector(
                method_name,
                X_train_bal,
                Y_train_bal,
                X_test_emb,
                top_k=DEFAULT_CANDIDATE_TOP_K
            )
            top_k_label = "VT"
        else:
            X_train_fs, X_test_fs, mask, selected_idx, scores = apply_statistical_selector(
                method_name,
                X_train_bal,
                Y_train_bal,
                X_test_emb,
                top_k=top_k
            )
            top_k_label = int(top_k)

        metrics, model, y_pred = train_evaluate_rf(
            X_train_fs,
            Y_train_bal,
            X_test_fs,
            Y_test,
            random_state=RANDOM_STATE
        )

        row = {
            "Experiment Group": "Statistical Selection",
            "Method": method_name,
            "Stage 1 Filter": method_name,
            "Top K": top_k_label,
            "Selected Features": int(X_train_fs.shape[1]),
            "Feature Ratio": float(X_train_fs.shape[1] / X_train_bal.shape[1]),
            **metrics
        }
        row["Research Score"] = compute_research_score(row)

        method_id = f"{method_name}|k={top_k_label}"
        statistical_results.append(row)
        all_results.append(row)
        statistical_artifacts[method_id] = {
            "experiment_group": "Statistical Selection",
            "method": method_name,
            "top_k": top_k_label,
            "mask": mask,
            "selected_idx": selected_idx,
            "scores": scores,
            "X_train": X_train_fs,
            "X_test": X_test_fs,
            "model": model,
            "y_pred": y_pred,
            "metrics": metrics,
            "selected_features": int(X_train_fs.shape[1])
        }

        print(f"{method_name} | Top K={top_k_label} | fitur={X_train_fs.shape[1]} | F1={metrics['F1 Weighted']:.4f} | Minority Recall={metrics['Minority Recall']:.4f}")

statistical_results_df = format_results_table(statistical_results)
statistical_results_df = statistical_results_df.sort_values(
    by=["Research Score", "F1 Weighted", "Minority Recall"],
    ascending=False
).reset_index(drop=True)

print("\nTabel hasil seleksi fitur statistik")
display(statistical_results_df[[
    "Experiment Group", "Method", "Top K", "Selected Features",
    "F1 Weighted", "F1 Macro", "Minority Recall", "Hamming Loss", "Research Score"
]])

BEST_STATISTICAL_METHOD = statistical_results_df.loc[0, "Method"]
BEST_STATISTICAL_TOP_K = statistical_results_df.loc[0, "Top K"]
BEST_STATISTICAL_ID = f"{BEST_STATISTICAL_METHOD}|k={BEST_STATISTICAL_TOP_K}"

print("Metode statistik terbaik untuk tahap penyaringan awal:", BEST_STATISTICAL_ID)


In [ ]:
# ============================================================
# 13. Fungsi objektif dan utilitas untuk IA-MPSO
# ============================================================
def sigmoid(x):
    x = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x))


def compute_redundancy_score(X_selected, max_features=250, random_state=42, eps=1e-12):
    # Redundansi dihitung sebagai rata-rata korelasi absolut antar fitur terpilih.
    # Jika fitur terlalu banyak, sebagian fitur diambil sebagai sampel agar proses tetap ringan.
    X_selected = np.asarray(X_selected, dtype=np.float64)
    n_features = X_selected.shape[1]
    if n_features <= 1:
        return 0.0

    rng = np.random.default_rng(random_state)
    if n_features > max_features:
        sampled_idx = rng.choice(n_features, size=max_features, replace=False)
        X_selected = X_selected[:, sampled_idx]
        n_features = X_selected.shape[1]

    std = X_selected.std(axis=0)
    valid = std > eps
    X_selected = X_selected[:, valid]
    n_features = X_selected.shape[1]
    if n_features <= 1:
        return 0.0

    corr = np.corrcoef(X_selected, rowvar=False)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    upper = np.triu_indices_from(corr, k=1)
    return float(np.mean(np.abs(corr[upper])))


def repair_mask(mask, rng, relevance_scores=None, min_features=20, max_ratio=0.70):
    mask = np.asarray(mask).astype(bool)
    n_features = len(mask)
    max_features = max(min_features, int(np.ceil(max_ratio * n_features)))

    if relevance_scores is None:
        relevance_scores = np.ones(n_features)
    relevance_scores = normalize_scores(relevance_scores)

    selected_count = int(mask.sum())

    if selected_count < min_features:
        needed = min_features - selected_count
        candidate_idx = np.where(~mask)[0]
        if len(candidate_idx) > 0:
            order = candidate_idx[np.argsort(relevance_scores[candidate_idx])[::-1]]
            mask[order[:needed]] = True

    if mask.sum() == 0:
        mask[rng.integers(0, n_features)] = True

    if mask.sum() > max_features:
        selected_idx = np.where(mask)[0]
        order = selected_idx[np.argsort(relevance_scores[selected_idx])[::-1]]
        keep_idx = order[:max_features]
        new_mask = np.zeros(n_features, dtype=bool)
        new_mask[keep_idx] = True
        mask = new_mask

    return mask


def make_mask_key(mask):
    return np.packbits(np.asarray(mask).astype(bool)).tobytes()


def subset_objective_function(
    feature_mask,
    X,
    Y,
    minority_label_indices,
    relevance_scores=None,
    cv_splits=3,
    random_state=42,
    cache=None,
    mode="proposed"
):
    feature_mask = np.asarray(feature_mask).astype(bool)
    if feature_mask.sum() == 0:
        return -9999.0

    cache = {} if cache is None else cache
    mask_key = (mode, make_mask_key(feature_mask))
    if mask_key in cache:
        return cache[mask_key]

    X_selected = X[:, feature_mask]
    n_splits = min(cv_splits, len(Y))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    weighted_f1_scores = []
    macro_f1_scores = []
    hamming_losses = []
    minority_recalls = []

    for train_idx, val_idx in kf.split(X_selected):
        X_train_fold = X_selected[train_idx]
        X_val_fold = X_selected[val_idx]
        Y_train_fold = Y[train_idx]
        Y_val_fold = Y[val_idx]

        base_rf = RandomForestClassifier(
            n_estimators=FITNESS_RF_N_ESTIMATORS,
            max_depth=FITNESS_RF_MAX_DEPTH,
            class_weight="balanced_subsample",
            random_state=random_state,
            n_jobs=-1
        )
        clf = MultiOutputClassifier(base_rf)
        clf.fit(X_train_fold, Y_train_fold)
        Y_pred_fold = clf.predict(X_val_fold)

        weighted_f1_scores.append(f1_score(Y_val_fold, Y_pred_fold, average="weighted", zero_division=0))
        macro_f1_scores.append(f1_score(Y_val_fold, Y_pred_fold, average="macro", zero_division=0))
        hamming_losses.append(hamming_loss(Y_val_fold, Y_pred_fold))

        recall_each_label = recall_score(Y_val_fold, Y_pred_fold, average=None, zero_division=0)
        minority_recalls.append(np.mean(recall_each_label[minority_label_indices]))

    weighted_f1 = float(np.mean(weighted_f1_scores))
    macro_f1 = float(np.mean(macro_f1_scores))
    hamming = float(np.mean(hamming_losses))
    minority_recall = float(np.mean(minority_recalls))
    feature_ratio = float(feature_mask.sum() / len(feature_mask))

    if mode == "standard":
        fitness = weighted_f1 - hamming
    else:
        redundancy = compute_redundancy_score(X_selected, random_state=random_state)
        fitness = (
            OBJ_W_WEIGHTED_F1 * weighted_f1
            + OBJ_W_MACRO_F1 * macro_f1
            + OBJ_W_MINOR_RECALL * minority_recall
            - OBJ_W_HAMMING * hamming
            - OBJ_W_SUBSET_SIZE * feature_ratio
            - OBJ_W_REDUNDANCY * redundancy
        )

    cache[mask_key] = float(fitness)
    return float(fitness)


print("Fungsi objektif IA-MPSO siap digunakan.")


In [ ]:
# ============================================================
# 14. Standard PSO sebagai pembanding metaheuristik
# ============================================================
def standard_pso_feature_selection(
    X,
    Y,
    pop_size=12,
    max_iter=15,
    random_state=42,
    patience=5
):
    rng = np.random.default_rng(random_state)
    n_features = X.shape[1]
    relevance_scores = np.ones(n_features)
    fitness_cache = {}

    positions = rng.integers(0, 2, size=(pop_size, n_features)).astype(bool)
    velocities = rng.normal(0, 1, size=(pop_size, n_features))
    positions = np.array([
        repair_mask(pos, rng, relevance_scores, min_features=MIN_SELECTED_FEATURES, max_ratio=1.00)
        for pos in positions
    ])

    pbest_positions = positions.copy()
    pbest_fitness = np.array([
        subset_objective_function(
            pos,
            X,
            Y,
            MINORITY_LABEL_INDICES,
            relevance_scores=relevance_scores,
            cv_splits=CV_SPLITS,
            random_state=random_state,
            cache=fitness_cache,
            mode="standard"
        )
        for pos in pbest_positions
    ])

    best_idx = int(np.argmax(pbest_fitness))
    gbest_position = pbest_positions[best_idx].copy()
    gbest_fitness = float(pbest_fitness[best_idx])
    no_improvement = 0
    history = []

    for iteration in range(max_iter):
        w = W_MAX - ((W_MAX - W_MIN) * iteration / max(1, max_iter - 1))
        improved = False

        for j in range(pop_size):
            r1 = rng.random(n_features)
            r2 = rng.random(n_features)
            velocities[j] = (
                w * velocities[j]
                + C1 * r1 * (pbest_positions[j].astype(int) - positions[j].astype(int))
                + C2 * r2 * (gbest_position.astype(int) - positions[j].astype(int))
            )

            probabilities = sigmoid(velocities[j])
            positions[j] = rng.random(n_features) < probabilities
            positions[j] = repair_mask(
                positions[j],
                rng,
                relevance_scores,
                min_features=MIN_SELECTED_FEATURES,
                max_ratio=1.00
            )

            current_fitness = subset_objective_function(
                positions[j],
                X,
                Y,
                MINORITY_LABEL_INDICES,
                relevance_scores=relevance_scores,
                cv_splits=CV_SPLITS,
                random_state=random_state,
                cache=fitness_cache,
                mode="standard"
            )

            if current_fitness > pbest_fitness[j]:
                pbest_positions[j] = positions[j].copy()
                pbest_fitness[j] = current_fitness

            if current_fitness > gbest_fitness:
                gbest_position = positions[j].copy()
                gbest_fitness = current_fitness
                improved = True

        history.append({
            "Iteration": iteration + 1,
            "Fitness": gbest_fitness,
            "Selected Features": int(gbest_position.sum())
        })
        no_improvement = 0 if improved else no_improvement + 1
        print(f"Standard PSO iter {iteration + 1:02d} | fitness={gbest_fitness:.4f} | fitur={gbest_position.sum()}")

        if no_improvement >= patience:
            break

    return gbest_position.astype(bool), gbest_fitness, pd.DataFrame(history)


print("Standard PSO siap digunakan.")


In [ ]:
# ============================================================
# 15. Imbalance-Aware Modified PSO sebagai metode usulan
# ============================================================
def imbalance_aware_modified_pso_feature_selection(
    X,
    Y,
    relevance_scores,
    pop_size=12,
    max_iter=15,
    random_state=42,
    patience=5,
    min_features=20,
    max_ratio=0.70
):
    rng = np.random.default_rng(random_state)
    n_features = X.shape[1]
    relevance_scores = normalize_scores(relevance_scores)
    fitness_cache = {}

    # Inisialisasi partikel dibuat bias pada fitur yang relevan terhadap label minoritas.
    base_prob = 0.10 + 0.65 * relevance_scores
    positions = rng.random((pop_size, n_features)) < base_prob
    velocities = rng.normal(0, 1, size=(pop_size, n_features))
    positions = np.array([
        repair_mask(pos, rng, relevance_scores, min_features=min_features, max_ratio=max_ratio)
        for pos in positions
    ])

    pbest_positions = positions.copy()
    pbest_fitness = np.array([
        subset_objective_function(
            pos,
            X,
            Y,
            MINORITY_LABEL_INDICES,
            relevance_scores=relevance_scores,
            cv_splits=CV_SPLITS,
            random_state=random_state,
            cache=fitness_cache,
            mode="proposed"
        )
        for pos in pbest_positions
    ])

    best_idx = int(np.argmax(pbest_fitness))
    gbest_position = pbest_positions[best_idx].copy()
    gbest_fitness = float(pbest_fitness[best_idx])
    no_improvement = 0
    history = []

    for iteration in range(max_iter):
        # Inertia weight dinamis untuk menjaga eksplorasi di awal dan eksploitasi di akhir.
        w = W_MAX - ((W_MAX - W_MIN) * iteration / max(1, max_iter - 1))
        improved = False

        for j in range(pop_size):
            r1 = rng.random(n_features)
            r2 = rng.random(n_features)

            velocities[j] = (
                w * velocities[j]
                + C1 * r1 * (pbest_positions[j].astype(int) - positions[j].astype(int))
                + C2 * r2 * (gbest_position.astype(int) - positions[j].astype(int))
            )

            # Relevance bias menjaga fitur penting untuk label minoritas tetap berpeluang tinggi dipilih.
            probabilities = sigmoid(velocities[j] + RELEVANCE_BIAS * (relevance_scores - 0.5))
            positions[j] = rng.random(n_features) < probabilities
            positions[j] = repair_mask(
                positions[j],
                rng,
                relevance_scores,
                min_features=min_features,
                max_ratio=max_ratio
            )

            current_fitness = subset_objective_function(
                positions[j],
                X,
                Y,
                MINORITY_LABEL_INDICES,
                relevance_scores=relevance_scores,
                cv_splits=CV_SPLITS,
                random_state=random_state,
                cache=fitness_cache,
                mode="proposed"
            )

            if current_fitness > pbest_fitness[j]:
                pbest_positions[j] = positions[j].copy()
                pbest_fitness[j] = current_fitness

            if current_fitness > gbest_fitness:
                gbest_position = positions[j].copy()
                gbest_fitness = current_fitness
                improved = True

        history.append({
            "Iteration": iteration + 1,
            "Fitness": gbest_fitness,
            "Selected Features": int(gbest_position.sum())
        })
        no_improvement = 0 if improved else no_improvement + 1
        print(f"IA-MPSO iter {iteration + 1:02d} | fitness={gbest_fitness:.4f} | fitur={gbest_position.sum()}")

        if no_improvement >= patience:
            break

    return gbest_position.astype(bool), gbest_fitness, pd.DataFrame(history)


print("Imbalance-Aware Modified PSO siap digunakan.")


In [ ]:
# ============================================================
# 16. Jalankan Standard PSO dan Proposed IA-MPSO
# ============================================================
metaheuristic_results = []
metaheuristic_artifacts = {}

# ------------------------------------------------------------
# 16.1 Standard PSO langsung pada seluruh fitur
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("Menjalankan Standard PSO pada seluruh fitur")

std_mask, std_fitness, std_history = standard_pso_feature_selection(
    X_train_bal,
    Y_train_bal,
    pop_size=PSO_POP_SIZE,
    max_iter=PSO_MAX_ITER,
    random_state=RANDOM_STATE,
    patience=PSO_PATIENCE
)

std_idx = np.where(std_mask)[0]
X_train_std = X_train_bal[:, std_mask]
X_test_std = X_test_emb[:, std_mask]
std_metrics, std_model, std_pred = train_evaluate_rf(
    X_train_std,
    Y_train_bal,
    X_test_std,
    Y_test,
    random_state=RANDOM_STATE
)

std_row = {
    "Experiment Group": "Metaheuristic Baseline",
    "Method": "Standard PSO",
    "Stage 1 Filter": "None",
    "Top K": "All",
    "Selected Features": int(X_train_std.shape[1]),
    "Feature Ratio": float(X_train_std.shape[1] / X_train_bal.shape[1]),
    "Fitness": std_fitness,
    **std_metrics
}
std_row["Research Score"] = compute_research_score(std_row)
metaheuristic_results.append(std_row)
all_results.append(std_row)

metaheuristic_artifacts["Standard PSO"] = {
    "experiment_group": "Metaheuristic Baseline",
    "method": "Standard PSO",
    "stage1_filter": "None",
    "selected_idx": std_idx,
    "mask": std_mask,
    "fitness": std_fitness,
    "history": std_history,
    "X_train": X_train_std,
    "X_test": X_test_std,
    "model": std_model,
    "y_pred": std_pred,
    "metrics": std_metrics,
    "selected_features": int(X_train_std.shape[1])
}

# ------------------------------------------------------------
# 16.2 Metode usulan: Statistical Screening + IA-MPSO
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("Menjalankan metode usulan: Statistical Screening + IA-MPSO")
print("Tahap penyaringan statistik yang digunakan:", BEST_STATISTICAL_ID)

stage1_artifact = statistical_artifacts[BEST_STATISTICAL_ID]
candidate_idx = np.asarray(stage1_artifact["selected_idx"])

# Jika metode statistik terbaik menghasilkan fitur terlalu banyak, ambil skor tertinggi sesuai DEFAULT_CANDIDATE_TOP_K.
if len(candidate_idx) > DEFAULT_CANDIDATE_TOP_K:
    stage1_scores = np.asarray(stage1_artifact["scores"])
    candidate_idx = np.argsort(stage1_scores)[::-1][:DEFAULT_CANDIDATE_TOP_K]
    candidate_idx = np.sort(candidate_idx)

X_train_candidate = X_train_bal[:, candidate_idx]
X_test_candidate = X_test_emb[:, candidate_idx]

# Relevance score lokal dihitung pada fitur kandidat.
# Jika skor statistik tersedia, skor tersebut dipakai sebagai bias awal IA-MPSO.
full_scores = np.asarray(stage1_artifact["scores"])
local_relevance_scores = normalize_scores(full_scores[candidate_idx])

proposed_mask_local, proposed_fitness, proposed_history = imbalance_aware_modified_pso_feature_selection(
    X_train_candidate,
    Y_train_bal,
    relevance_scores=local_relevance_scores,
    pop_size=PSO_POP_SIZE,
    max_iter=PSO_MAX_ITER,
    random_state=RANDOM_STATE,
    patience=PSO_PATIENCE,
    min_features=min(MIN_SELECTED_FEATURES, X_train_candidate.shape[1]),
    max_ratio=MAX_FEATURE_RATIO
)

proposed_selected_original_idx = candidate_idx[proposed_mask_local]
X_train_proposed = X_train_bal[:, proposed_selected_original_idx]
X_test_proposed = X_test_emb[:, proposed_selected_original_idx]

proposed_metrics, proposed_model, proposed_pred = train_evaluate_rf(
    X_train_proposed,
    Y_train_bal,
    X_test_proposed,
    Y_test,
    random_state=RANDOM_STATE
)

proposed_row = {
    "Experiment Group": "Proposed Method",
    "Method": "Imbalance-Aware Modified PSO",
    "Stage 1 Filter": BEST_STATISTICAL_METHOD,
    "Top K": int(len(candidate_idx)),
    "Selected Features": int(X_train_proposed.shape[1]),
    "Feature Ratio": float(X_train_proposed.shape[1] / X_train_bal.shape[1]),
    "Fitness": proposed_fitness,
    **proposed_metrics
}
proposed_row["Research Score"] = compute_research_score(proposed_row)
metaheuristic_results.append(proposed_row)
all_results.append(proposed_row)

metaheuristic_artifacts["Imbalance-Aware Modified PSO"] = {
    "experiment_group": "Proposed Method",
    "method": "Imbalance-Aware Modified PSO",
    "stage1_filter": BEST_STATISTICAL_METHOD,
    "stage1_id": BEST_STATISTICAL_ID,
    "candidate_idx": candidate_idx,
    "selected_idx": proposed_selected_original_idx,
    "local_mask": proposed_mask_local,
    "fitness": proposed_fitness,
    "history": proposed_history,
    "X_train": X_train_proposed,
    "X_test": X_test_proposed,
    "model": proposed_model,
    "y_pred": proposed_pred,
    "metrics": proposed_metrics,
    "selected_features": int(X_train_proposed.shape[1])
}

metaheuristic_results_df = format_results_table(metaheuristic_results)
metaheuristic_results_df["Fitness"] = metaheuristic_results_df["Fitness"].astype(float).round(4)
metaheuristic_results_df = metaheuristic_results_df.sort_values(
    by=["Research Score", "F1 Weighted", "Minority Recall"],
    ascending=False
).reset_index(drop=True)

print("\nTabel hasil metode metaheuristik")
display(metaheuristic_results_df[[
    "Experiment Group", "Method", "Stage 1 Filter", "Top K", "Selected Features",
    "F1 Weighted", "F1 Macro", "Minority Recall", "Hamming Loss", "Research Score"
]])


In [ ]:
# ============================================================
# 17. Tabel perbandingan utama seluruh eksperimen
# ============================================================
comparison_results_df = format_results_table(all_results)
comparison_results_df = comparison_results_df.sort_values(
    by=["Research Score", "F1 Weighted", "Minority Recall"],
    ascending=False
).reset_index(drop=True)

main_columns = [
    "Experiment Group",
    "Method",
    "Stage 1 Filter",
    "Top K",
    "Selected Features",
    "Feature Ratio",
    "Precision Weighted",
    "Recall Weighted",
    "F1 Weighted",
    "F1 Macro",
    "Minority Recall",
    "Minority F1",
    "Exact Match Accuracy",
    "Hamming Loss",
    "Research Score"
]

print("Tabel perbandingan utama")
display(comparison_results_df[main_columns])

BEST_OVERALL_GROUP = comparison_results_df.loc[0, "Experiment Group"]
BEST_OVERALL_METHOD = comparison_results_df.loc[0, "Method"]

if BEST_OVERALL_METHOD == "All Features":
    best_artifact = baseline_artifact
elif BEST_OVERALL_METHOD in statistical_results_df["Method"].values and BEST_OVERALL_GROUP == "Statistical Selection":
    best_top_k = comparison_results_df.loc[0, "Top K"]
    best_id = f"{BEST_OVERALL_METHOD}|k={best_top_k}"
    best_artifact = statistical_artifacts[best_id]
else:
    best_artifact = metaheuristic_artifacts[BEST_OVERALL_METHOD]

Y_pred_best = best_artifact["y_pred"]

print("Model terbaik berdasarkan Research Score:")
print("Kelompok eksperimen:", BEST_OVERALL_GROUP)
print("Metode:", BEST_OVERALL_METHOD)
print("Jumlah fitur digunakan:", best_artifact["selected_features"])
print("Label minoritas:", MINORITY_LABELS)


## Pengujian Perbandingan Classifier Multilabel

Bagian ini menambahkan pengujian antar classifier pada data training dan testing yang sama. Pengujian dilakukan menggunakan empat classifier, yaitu Random Forest MultiOutputClassifier, SVM, k-NN, dan Decision Tree.

Pengujian dilakukan pada beberapa skenario fitur agar hasil dapat menunjukkan pengaruh classifier dan jumlah fitur terhadap performa klasifikasi multilabel. SVM menggunakan LinearSVC agar proses eksperimen lebih stabil pada data embedding berdimensi tinggi.


In [ ]:
# ============================================================
# 19. Pengujian perbandingan classifier multilabel
# ============================================================
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

classifier_results = []
classifier_artifacts = {}

def prepare_features_for_classifier(X_train, X_test, classifier_name):
    """
    SVM dan k-NN lebih sensitif terhadap skala fitur.
    Oleh karena itu, kedua classifier tersebut menggunakan StandardScaler.
    Random Forest dan Decision Tree tidak memerlukan scaling.
    """
    if classifier_name in ["SVM", "k-NN"]:
        scaler = StandardScaler(with_mean=False)
        X_train_prepared = scaler.fit_transform(X_train)
        X_test_prepared = scaler.transform(X_test)
        return X_train_prepared, X_test_prepared, scaler

    return X_train, X_test, None


def build_multilabel_classifier(classifier_name, random_state=42):
    """
    Semua classifier dibungkus dengan MultiOutputClassifier agar setiap label
    dipelajari sebagai masalah klasifikasi biner yang terpisah.
    """
    if classifier_name == "Random Forest MultiOutputClassifier":
        base_clf = RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            class_weight="balanced_subsample",
            random_state=random_state,
            n_jobs=-1
        )

    elif classifier_name == "SVM":
        base_clf = LinearSVC(
            C=1.0,
            class_weight="balanced",
            max_iter=5000,
            random_state=random_state
        )

    elif classifier_name == "k-NN":
        n_neighbors = min(5, max(1, len(Y_train_bal) - 1))
        base_clf = KNeighborsClassifier(
            n_neighbors=n_neighbors,
            weights="distance",
            metric="minkowski"
        )

    elif classifier_name == "Decision Tree":
        base_clf = DecisionTreeClassifier(
            max_depth=RF_MAX_DEPTH,
            class_weight="balanced",
            random_state=random_state
        )

    else:
        raise ValueError(f"Classifier tidak dikenal: {classifier_name}")

    return MultiOutputClassifier(base_clf)


def train_evaluate_multilabel_classifier(
    classifier_name,
    X_train,
    Y_train,
    X_test,
    Y_test,
    random_state=42
):
    X_train_ready, X_test_ready, scaler = prepare_features_for_classifier(
        X_train,
        X_test,
        classifier_name
    )

    clf = build_multilabel_classifier(classifier_name, random_state=random_state)
    clf.fit(X_train_ready, Y_train)

    Y_pred = clf.predict(X_test_ready)
    metrics = compute_main_metrics(
        Y_test,
        Y_pred,
        minority_label_indices=MINORITY_LABEL_INDICES
    )

    return metrics, clf, Y_pred, scaler


# Skenario fitur yang diuji.
# Bagian ini dapat dikurangi jika waktu running terlalu lama.
classifier_feature_sets = {
    "All Features": {
        "Experiment Group": "Baseline",
        "Method": "All Features",
        "Stage 1 Filter": "None",
        "Top K": "All",
        "X_train": X_train_bal,
        "X_test": X_test_emb,
        "Selected Features": int(X_train_bal.shape[1]),
        "Feature Ratio": float(X_train_bal.shape[1] / X_train_bal.shape[1])
    },
    "Best Statistical Selection": {
        "Experiment Group": "Statistical Selection",
        "Method": BEST_STATISTICAL_METHOD,
        "Stage 1 Filter": BEST_STATISTICAL_METHOD,
        "Top K": BEST_STATISTICAL_TOP_K,
        "X_train": statistical_artifacts[BEST_STATISTICAL_ID]["X_train"],
        "X_test": statistical_artifacts[BEST_STATISTICAL_ID]["X_test"],
        "Selected Features": int(statistical_artifacts[BEST_STATISTICAL_ID]["selected_features"]),
        "Feature Ratio": float(statistical_artifacts[BEST_STATISTICAL_ID]["selected_features"] / X_train_bal.shape[1])
    },
    "Standard PSO": {
        "Experiment Group": "Metaheuristic Baseline",
        "Method": "Standard PSO",
        "Stage 1 Filter": "None",
        "Top K": "All",
        "X_train": metaheuristic_artifacts["Standard PSO"]["X_train"],
        "X_test": metaheuristic_artifacts["Standard PSO"]["X_test"],
        "Selected Features": int(metaheuristic_artifacts["Standard PSO"]["selected_features"]),
        "Feature Ratio": float(metaheuristic_artifacts["Standard PSO"]["selected_features"] / X_train_bal.shape[1])
    },
    "Proposed IA-MPSO": {
        "Experiment Group": "Proposed Method",
        "Method": "Imbalance-Aware Modified PSO",
        "Stage 1 Filter": BEST_STATISTICAL_METHOD,
        "Top K": int(len(metaheuristic_artifacts["Imbalance-Aware Modified PSO"]["candidate_idx"])),
        "X_train": metaheuristic_artifacts["Imbalance-Aware Modified PSO"]["X_train"],
        "X_test": metaheuristic_artifacts["Imbalance-Aware Modified PSO"]["X_test"],
        "Selected Features": int(metaheuristic_artifacts["Imbalance-Aware Modified PSO"]["selected_features"]),
        "Feature Ratio": float(metaheuristic_artifacts["Imbalance-Aware Modified PSO"]["selected_features"] / X_train_bal.shape[1])
    }
}

CLASSIFIERS_TO_TEST = [
    "Random Forest MultiOutputClassifier",
    "SVM",
    "k-NN",
    "Decision Tree"
]

for feature_scenario, feature_info in classifier_feature_sets.items():
    print("\n" + "=" * 70)
    print(f"Pengujian classifier pada skenario fitur: {feature_scenario}")
    print("Jumlah fitur:", feature_info["Selected Features"])

    for classifier_name in CLASSIFIERS_TO_TEST:
        print(f"Menjalankan classifier: {classifier_name}")

        metrics, clf, y_pred, scaler = train_evaluate_multilabel_classifier(
            classifier_name=classifier_name,
            X_train=feature_info["X_train"],
            Y_train=Y_train_bal,
            X_test=feature_info["X_test"],
            Y_test=Y_test,
            random_state=RANDOM_STATE
        )

        row = {
            "Feature Scenario": feature_scenario,
            "Experiment Group": feature_info["Experiment Group"],
            "Feature Selection Method": feature_info["Method"],
            "Stage 1 Filter": feature_info["Stage 1 Filter"],
            "Top K": feature_info["Top K"],
            "Classifier": classifier_name,
            "Selected Features": feature_info["Selected Features"],
            "Feature Ratio": feature_info["Feature Ratio"],
            **metrics
        }
        row["Research Score"] = compute_research_score(row)

        classifier_results.append(row)

        artifact_key = f"{feature_scenario}|{classifier_name}"
        classifier_artifacts[artifact_key] = {
            "feature_scenario": feature_scenario,
            "classifier": classifier_name,
            "model": clf,
            "scaler": scaler,
            "y_pred": y_pred,
            "metrics": metrics,
            "selected_features": feature_info["Selected Features"]
        }

        print(
            f"{classifier_name} | "
            f"F1 Weighted={metrics['F1 Weighted']:.4f} | "
            f"F1 Macro={metrics['F1 Macro']:.4f} | "
            f"Minority Recall={metrics['Minority Recall']:.4f} | "
            f"Hamming Loss={metrics['Hamming Loss']:.4f}"
        )

classifier_comparison_df = format_results_table(classifier_results)
classifier_comparison_df = classifier_comparison_df.sort_values(
    by=["Feature Scenario", "Research Score", "F1 Weighted", "Minority Recall"],
    ascending=[True, False, False, False]
).reset_index(drop=True)

classifier_main_columns = [
    "Feature Scenario",
    "Experiment Group",
    "Feature Selection Method",
    "Stage 1 Filter",
    "Top K",
    "Classifier",
    "Selected Features",
    "Feature Ratio",
    "Precision Weighted",
    "Recall Weighted",
    "F1 Weighted",
    "F1 Macro",
    "Minority Recall",
    "Minority F1",
    "Exact Match Accuracy",
    "Hamming Loss",
    "Research Score"
]

print("\nTabel perbandingan hasil antar classifier")
display(classifier_comparison_df[classifier_main_columns])

best_classifier_row = classifier_comparison_df.sort_values(
    by=["Research Score", "F1 Weighted", "Minority Recall"],
    ascending=False
).iloc[0]

print("\nClassifier terbaik berdasarkan Research Score:")
print("Skenario fitur:", best_classifier_row["Feature Scenario"])
print("Metode seleksi fitur:", best_classifier_row["Feature Selection Method"])
print("Classifier:", best_classifier_row["Classifier"])
print("Jumlah fitur:", best_classifier_row["Selected Features"])
print("Research Score:", best_classifier_row["Research Score"])


## Pengujian Uji Signifikansi dan Paired t-test

Bagian ini menambahkan uji signifikansi setelah pengujian classifier. Uji dilakukan secara berpasangan karena setiap metode diuji pada data testing yang sama.

Pengujian terdiri atas tiga bagian:

1. **Uji normalitas Shapiro-Wilk** pada selisih skor antar metode.
2. **Paired sample t-test** apabila selisih skor berdistribusi normal.
3. **Wilcoxon signed-rank test** apabila selisih skor tidak berdistribusi normal.

Selain itu, bagian ini menambahkan **bootstrap paired test** untuk Research Score. Pendekatan ini digunakan karena Research Score pada tabel utama hanya berupa satu nilai agregat, sehingga tidak dapat langsung diuji dengan t-test tanpa data per fold atau per running.

In [ ]:
# ============================================================
# 20A. Uji signifikansi, paired t-test, Wilcoxon, dan bootstrap Research Score
# ============================================================

try:
    from scipy import stats
except Exception:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scipy"])
    from scipy import stats

SIGNIFICANCE_ALPHA = 0.05
BOOTSTRAP_N = 500
BOOTSTRAP_RANDOM_STATE = RANDOM_STATE

def sample_hamming_accuracy_scores(Y_true, Y_pred):
    """
    Skor per data untuk multilabel.
    Nilai 1 berarti semua label pada data tersebut benar.
    Nilai mendekati 0 berarti banyak label salah.
    """
    Y_true = np.asarray(Y_true).astype(int)
    Y_pred = np.asarray(Y_pred).astype(int)
    return 1.0 - np.mean(Y_true != Y_pred, axis=1)


def sample_exact_match_scores(Y_true, Y_pred):
    """
    Skor per data berdasarkan kecocokan seluruh label.
    Nilai 1 jika seluruh label benar, 0 jika ada minimal satu label salah.
    """
    Y_true = np.asarray(Y_true).astype(int)
    Y_pred = np.asarray(Y_pred).astype(int)
    return np.all(Y_true == Y_pred, axis=1).astype(float)


def sample_f1_scores(Y_true, Y_pred):
    """
    F1-score per data untuk kasus multilabel.
    Jika data aktual dan prediksi sama-sama tidak memiliki label positif,
    skor diberi nilai 1 karena prediksi kosong sesuai dengan aktual kosong.
    """
    Y_true = np.asarray(Y_true).astype(int)
    Y_pred = np.asarray(Y_pred).astype(int)

    tp = np.sum((Y_true == 1) & (Y_pred == 1), axis=1)
    fp = np.sum((Y_true == 0) & (Y_pred == 1), axis=1)
    fn = np.sum((Y_true == 1) & (Y_pred == 0), axis=1)

    denom = (2 * tp) + fp + fn
    scores = np.where(denom == 0, 1.0, (2 * tp) / denom)
    return scores.astype(float)


def paired_significance_test(proposed_scores, comparator_scores, alpha=0.05):
    """
    Melakukan uji normalitas, paired t-test, dan Wilcoxon.
    Hipotesis arah:
    H0: skor IA-MPSO tidak lebih tinggi daripada metode pembanding.
    H1: skor IA-MPSO lebih tinggi daripada metode pembanding.
    """
    proposed_scores = np.asarray(proposed_scores, dtype=float)
    comparator_scores = np.asarray(comparator_scores, dtype=float)

    valid_mask = ~(np.isnan(proposed_scores) | np.isnan(comparator_scores))
    proposed_scores = proposed_scores[valid_mask]
    comparator_scores = comparator_scores[valid_mask]
    diff = proposed_scores - comparator_scores

    n = len(diff)
    mean_proposed = float(np.mean(proposed_scores)) if n > 0 else np.nan
    mean_comparator = float(np.mean(comparator_scores)) if n > 0 else np.nan
    mean_diff = float(np.mean(diff)) if n > 0 else np.nan
    std_diff = float(np.std(diff, ddof=1)) if n > 1 else np.nan

    if n >= 3 and np.isfinite(std_diff) and std_diff > 0:
        shapiro_stat, shapiro_p = stats.shapiro(diff)
        shapiro_stat = float(shapiro_stat)
        shapiro_p = float(shapiro_p)
    else:
        shapiro_stat, shapiro_p = np.nan, np.nan

    # Paired t-test satu arah: IA-MPSO > pembanding.
    if n >= 2 and np.isfinite(std_diff) and std_diff > 0:
        try:
            t_stat, ttest_p = stats.ttest_rel(
                proposed_scores,
                comparator_scores,
                alternative="greater"
            )
        except TypeError:
            # Fallback untuk SciPy lama.
            t_stat_two, p_two = stats.ttest_rel(proposed_scores, comparator_scores)
            t_stat = t_stat_two
            ttest_p = p_two / 2 if mean_diff > 0 else 1 - (p_two / 2)
        t_stat = float(t_stat)
        ttest_p = float(ttest_p)
        cohen_dz = float(mean_diff / std_diff)
    else:
        t_stat, ttest_p, cohen_dz = np.nan, np.nan, np.nan

    # Wilcoxon satu arah: IA-MPSO > pembanding.
    if n >= 2 and np.any(diff != 0):
        try:
            wilcoxon_stat, wilcoxon_p = stats.wilcoxon(
                proposed_scores,
                comparator_scores,
                alternative="greater",
                zero_method="wilcox"
            )
        except Exception:
            wilcoxon_stat, wilcoxon_p = np.nan, np.nan
        wilcoxon_stat = float(wilcoxon_stat) if np.isfinite(wilcoxon_stat) else np.nan
        wilcoxon_p = float(wilcoxon_p) if np.isfinite(wilcoxon_p) else np.nan
    else:
        wilcoxon_stat, wilcoxon_p = np.nan, np.nan

    if np.isfinite(shapiro_p):
        normality = "Normal" if shapiro_p >= alpha else "Tidak normal"
        recommended_test = "Paired t-test" if shapiro_p >= alpha else "Wilcoxon signed-rank"
    else:
        normality = "Tidak diuji"
        recommended_test = "Wilcoxon signed-rank"

    if recommended_test == "Paired t-test":
        recommended_p = ttest_p
    else:
        recommended_p = wilcoxon_p

    if np.isfinite(recommended_p):
        decision = "Signifikan" if recommended_p < alpha else "Tidak signifikan"
    else:
        decision = "Tidak dapat ditentukan"

    return {
        "N": n,
        "Mean Proposed": mean_proposed,
        "Mean Comparator": mean_comparator,
        "Mean Difference": mean_diff,
        "Std Difference": std_diff,
        "Shapiro Statistic": shapiro_stat,
        "Shapiro p-value": shapiro_p,
        "Normality": normality,
        "t Statistic": t_stat,
        "Paired t-test p-value": ttest_p,
        "Wilcoxon Statistic": wilcoxon_stat,
        "Wilcoxon p-value": wilcoxon_p,
        "Recommended Test": recommended_test,
        "Recommended p-value": recommended_p,
        "Decision": decision,
        "Cohen dz": cohen_dz
    }


def get_classifier_result_row(feature_scenario, classifier_name):
    row = classifier_comparison_df[
        (classifier_comparison_df["Feature Scenario"] == feature_scenario)
        & (classifier_comparison_df["Classifier"] == classifier_name)
    ]
    if row.empty:
        raise ValueError(f"Baris hasil tidak ditemukan: {feature_scenario} | {classifier_name}")
    return row.iloc[0].to_dict()


def compute_research_score_from_prediction(Y_true_subset, Y_pred_subset, feature_ratio):
    metrics = compute_main_metrics(
        Y_true_subset,
        Y_pred_subset,
        minority_label_indices=MINORITY_LABEL_INDICES
    )
    row = {**metrics, "Feature Ratio": feature_ratio}
    return float(compute_research_score(row))


def bootstrap_research_score_test(
    Y_true,
    proposed_pred,
    comparator_pred,
    proposed_feature_ratio,
    comparator_feature_ratio,
    n_bootstrap=500,
    random_state=42,
    alpha=0.05
):
    """
    Bootstrap paired test untuk Research Score.
    H0: selisih Research Score IA-MPSO - pembanding <= 0.
    H1: selisih Research Score IA-MPSO - pembanding > 0.
    """
    rng = np.random.default_rng(random_state)
    Y_true = np.asarray(Y_true).astype(int)
    proposed_pred = np.asarray(proposed_pred).astype(int)
    comparator_pred = np.asarray(comparator_pred).astype(int)

    n = len(Y_true)
    diff_scores = []

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        score_proposed = compute_research_score_from_prediction(
            Y_true[idx],
            proposed_pred[idx],
            proposed_feature_ratio
        )
        score_comparator = compute_research_score_from_prediction(
            Y_true[idx],
            comparator_pred[idx],
            comparator_feature_ratio
        )
        diff_scores.append(score_proposed - score_comparator)

    diff_scores = np.asarray(diff_scores, dtype=float)
    mean_diff = float(np.mean(diff_scores))
    ci_low, ci_high = np.percentile(diff_scores, [2.5, 97.5])

    # p-value satu arah untuk H1: IA-MPSO lebih baik.
    p_value = float((np.sum(diff_scores <= 0) + 1) / (n_bootstrap + 1))
    decision = "Signifikan" if p_value < alpha else "Tidak signifikan"

    return {
        "Bootstrap N": int(n_bootstrap),
        "Mean Difference": mean_diff,
        "CI 2.5%": float(ci_low),
        "CI 97.5%": float(ci_high),
        "Bootstrap p-value": p_value,
        "Decision": decision
    }


COMPARATOR_SCENARIOS = [
    "All Features",
    "Best Statistical Selection",
    "Standard PSO"
]

SAMPLE_METRIC_FUNCTIONS = {
    "Sample F1": sample_f1_scores,
    "Sample Hamming Accuracy": sample_hamming_accuracy_scores,
    "Sample Exact Match": sample_exact_match_scores
}

significance_rows = []
bootstrap_rows = []

for classifier_name in CLASSIFIERS_TO_TEST:
    proposed_key = f"Proposed IA-MPSO|{classifier_name}"

    if proposed_key not in classifier_artifacts:
        print(f"Dilewati karena artifact tidak tersedia: {proposed_key}")
        continue

    proposed_pred = classifier_artifacts[proposed_key]["y_pred"]
    proposed_row = get_classifier_result_row("Proposed IA-MPSO", classifier_name)
    proposed_feature_ratio = float(proposed_row["Feature Ratio"])

    for comparator_scenario in COMPARATOR_SCENARIOS:
        comparator_key = f"{comparator_scenario}|{classifier_name}"

        if comparator_key not in classifier_artifacts:
            print(f"Dilewati karena artifact tidak tersedia: {comparator_key}")
            continue

        comparator_pred = classifier_artifacts[comparator_key]["y_pred"]
        comparator_row = get_classifier_result_row(comparator_scenario, classifier_name)
        comparator_feature_ratio = float(comparator_row["Feature Ratio"])

        for metric_name, metric_func in SAMPLE_METRIC_FUNCTIONS.items():
            proposed_scores = metric_func(Y_test, proposed_pred)
            comparator_scores = metric_func(Y_test, comparator_pred)
            test_result = paired_significance_test(
                proposed_scores,
                comparator_scores,
                alpha=SIGNIFICANCE_ALPHA
            )

            significance_rows.append({
                "Classifier": classifier_name,
                "Proposed Scenario": "Proposed IA-MPSO",
                "Comparator Scenario": comparator_scenario,
                "Metric Tested": metric_name,
                **test_result
            })

        bootstrap_result = bootstrap_research_score_test(
            Y_true=Y_test,
            proposed_pred=proposed_pred,
            comparator_pred=comparator_pred,
            proposed_feature_ratio=proposed_feature_ratio,
            comparator_feature_ratio=comparator_feature_ratio,
            n_bootstrap=BOOTSTRAP_N,
            random_state=BOOTSTRAP_RANDOM_STATE,
            alpha=SIGNIFICANCE_ALPHA
        )

        bootstrap_rows.append({
            "Classifier": classifier_name,
            "Proposed Scenario": "Proposed IA-MPSO",
            "Comparator Scenario": comparator_scenario,
            "Metric Tested": "Research Score",
            "Proposed Research Score": float(proposed_row["Research Score"]),
            "Comparator Research Score": float(comparator_row["Research Score"]),
            **bootstrap_result
        })

significance_results_df = pd.DataFrame(significance_rows)
bootstrap_research_score_df = pd.DataFrame(bootstrap_rows)

round_cols_significance = [
    "Mean Proposed", "Mean Comparator", "Mean Difference", "Std Difference",
    "Shapiro Statistic", "Shapiro p-value", "t Statistic", "Paired t-test p-value",
    "Wilcoxon Statistic", "Wilcoxon p-value", "Recommended p-value", "Cohen dz"
]
for col in round_cols_significance:
    if col in significance_results_df.columns:
        significance_results_df[col] = significance_results_df[col].astype(float).round(6)

round_cols_bootstrap = [
    "Proposed Research Score", "Comparator Research Score", "Mean Difference",
    "CI 2.5%", "CI 97.5%", "Bootstrap p-value"
]
for col in round_cols_bootstrap:
    if col in bootstrap_research_score_df.columns:
        bootstrap_research_score_df[col] = bootstrap_research_score_df[col].astype(float).round(6)

print("Tabel uji normalitas, paired t-test, dan Wilcoxon")
display(significance_results_df)

print("Tabel bootstrap paired test untuk Research Score")
display(bootstrap_research_score_df)

significance_results_df.to_csv("hasil_uji_signifikansi_paired_test.csv", index=False)
bootstrap_research_score_df.to_csv("hasil_uji_bootstrap_research_score.csv", index=False)

with pd.ExcelWriter("hasil_uji_signifikansi_IA_MPSO.xlsx", engine="openpyxl") as writer:
    significance_results_df.to_excel(writer, sheet_name="Paired Test", index=False)
    bootstrap_research_score_df.to_excel(writer, sheet_name="Bootstrap Score", index=False)

print("File uji signifikansi berhasil disimpan:")
print("1. hasil_uji_signifikansi_paired_test.csv")
print("2. hasil_uji_bootstrap_research_score.csv")
print("3. hasil_uji_signifikansi_IA_MPSO.xlsx")


In [ ]:
# ============================================================
# 20. Evaluasi per label untuk model terbaik
# ============================================================
precision_per_label = precision_score(Y_test, Y_pred_best, average=None, zero_division=0)
recall_per_label = recall_score(Y_test, Y_pred_best, average=None, zero_division=0)
f1_per_label = f1_score(Y_test, Y_pred_best, average=None, zero_division=0)
accuracy_per_label = [accuracy_score(Y_test[:, i], Y_pred_best[:, i]) for i in range(len(LABELS))]

per_label_df = pd.DataFrame({
    "Label": LABELS,
    "Is Minority Label": [label in MINORITY_LABELS for label in LABELS],
    "Positive Count Train Original": Y_train.sum(axis=0),
    "Positive Count Test": Y_test.sum(axis=0),
    "Precision": precision_per_label,
    "Recall": recall_per_label,
    "F1 score": f1_per_label,
    "Accuracy": accuracy_per_label
})

for col in ["Precision", "Recall", "F1 score", "Accuracy"]:
    per_label_df[col] = per_label_df[col].round(4)

print("Evaluasi per label untuk model terbaik:")
display(per_label_df)

plt.figure(figsize=(10, 5))
bars = plt.bar(per_label_df["Label"], per_label_df["Recall"])
plt.title("Recall per Label pada Model Terbaik")
plt.xlabel("Label")
plt.ylabel("Recall")
plt.ylim(0, 1.05)
plt.xticks(rotation=45)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.2f}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 21. Analisis fitur terpilih dan efisiensi subset
# ============================================================
selected_feature_df = pd.DataFrame({
    "Selected Feature Original Index": best_artifact["selected_idx"]
})

summary_efficiency_df = comparison_results_df[[
    "Experiment Group",
    "Method",
    "Selected Features",
    "Feature Ratio",
    "F1 Weighted",
    "F1 Macro",
    "Minority Recall",
    "Hamming Loss",
    "Research Score"
]].copy()

print("Ringkasan efisiensi fitur:")
display(summary_efficiency_df)

print("Contoh indeks fitur terpilih pada model terbaik:")
display(selected_feature_df.head(20))


In [ ]:
# ============================================================
# 23. Simpan hasil evaluasi, uji signifikansi, fitur terpilih, dan model terbaik
# ============================================================
best_model = best_artifact["model"]
best_selected_original_idx = best_artifact["selected_idx"]

model_package = {
    "embedding_backend": EMBEDDING_BACKEND,
    "model_name": MODEL_NAME,
    "tfidf_vectorizer": TFIDF_VECTORIZER,
    "labels": LABELS,
    "minority_labels": MINORITY_LABELS,
    "minority_label_indices": MINORITY_LABEL_INDICES,
    "text_column": TEXT_COL,
    "clean_text_column": CLEAN_TEXT_COL,
    "split_random_state": SPLIT_RANDOM_STATE,
    "best_overall_group": BEST_OVERALL_GROUP,
    "best_overall_method": BEST_OVERALL_METHOD,
    "best_statistical_id_for_stage1": BEST_STATISTICAL_ID,
    "best_selected_original_idx": best_selected_original_idx,
    "best_selected_feature_count": int(len(best_selected_original_idx)),
    "classifier": best_model,
    "label_distribution": label_distribution_df,
    "statistical_results": statistical_results_df,
    "metaheuristic_results": metaheuristic_results_df,
    "comparison_results": comparison_results_df,
    "classifier_comparison_results": classifier_comparison_df,
    "per_label_results": per_label_df,
    "selected_feature_df": selected_feature_df,
    "significance_results": significance_results_df if "significance_results_df" in globals() else None,
    "bootstrap_research_score": bootstrap_research_score_df if "bootstrap_research_score_df" in globals() else None,
    "config": {
        "test_size": TEST_SIZE,
        "feature_top_k_list": FEATURE_TOP_K_LIST,
        "default_candidate_top_k": DEFAULT_CANDIDATE_TOP_K,
        "min_selected_features": MIN_SELECTED_FEATURES,
        "max_feature_ratio": MAX_FEATURE_RATIO,
        "rf_n_estimators": RF_N_ESTIMATORS,
        "rf_max_depth": RF_MAX_DEPTH,
        "fitness_rf_n_estimators": FITNESS_RF_N_ESTIMATORS,
        "fitness_rf_max_depth": FITNESS_RF_MAX_DEPTH,
        "cv_splits": CV_SPLITS,
        "pso_pop_size": PSO_POP_SIZE,
        "pso_max_iter": PSO_MAX_ITER,
        "pso_patience": PSO_PATIENCE,
        "random_state": RANDOM_STATE,
        "objective_weights": {
            "weighted_f1": OBJ_W_WEIGHTED_F1,
            "macro_f1": OBJ_W_MACRO_F1,
            "minority_recall": OBJ_W_MINOR_RECALL,
            "hamming_loss": OBJ_W_HAMMING,
            "subset_size": OBJ_W_SUBSET_SIZE,
            "redundancy": OBJ_W_REDUNDANCY
        }
    }
}

joblib.dump(model_package, "best_rf_multilabel_model_IA_MPSO.joblib")

statistical_results_df.to_csv("hasil_seleksi_fitur_statistik.csv", index=False)
metaheuristic_results_df.to_csv("hasil_seleksi_fitur_metaheuristik.csv", index=False)
comparison_results_df.to_csv("hasil_perbandingan_semua_metode.csv", index=False)
classifier_comparison_df.to_csv("hasil_perbandingan_antar_classifier.csv", index=False)
per_label_df.to_csv("hasil_per_label_model_terbaik.csv", index=False)
selected_feature_df.to_csv("fitur_terpilih_model_terbaik.csv", index=False)
label_distribution_df.to_csv("distribusi_label.csv", index=False)

if "significance_results_df" in globals():
    significance_results_df.to_csv("hasil_uji_signifikansi_paired_test.csv", index=False)

if "bootstrap_research_score_df" in globals():
    bootstrap_research_score_df.to_csv("hasil_uji_bootstrap_research_score.csv", index=False)

with pd.ExcelWriter("hasil_eksperimen_IA_MPSO.xlsx", engine="openpyxl") as writer:
    label_distribution_df.to_excel(writer, sheet_name="Distribusi Label", index=False)
    statistical_results_df.to_excel(writer, sheet_name="Statistik", index=False)
    metaheuristic_results_df.to_excel(writer, sheet_name="Metaheuristik", index=False)
    comparison_results_df.to_excel(writer, sheet_name="Perbandingan", index=False)
    classifier_comparison_df.to_excel(writer, sheet_name="Classifier", index=False)
    per_label_df.to_excel(writer, sheet_name="Per Label", index=False)
    selected_feature_df.to_excel(writer, sheet_name="Fitur Terpilih", index=False)

    if "significance_results_df" in globals():
        significance_results_df.to_excel(writer, sheet_name="Uji Signifikansi", index=False)

    if "bootstrap_research_score_df" in globals():
        bootstrap_research_score_df.to_excel(writer, sheet_name="Bootstrap Score", index=False)

print("File berhasil disimpan:")
print("1. best_rf_multilabel_model_IA_MPSO.joblib")
print("2. hasil_eksperimen_IA_MPSO.xlsx")
print("3. hasil_seleksi_fitur_statistik.csv")
print("4. hasil_seleksi_fitur_metaheuristik.csv")
print("5. hasil_perbandingan_semua_metode.csv")
print("6. hasil_perbandingan_antar_classifier.csv")
print("7. hasil_per_label_model_terbaik.csv")
print("8. fitur_terpilih_model_terbaik.csv")
print("9. distribusi_label.csv")
if "significance_results_df" in globals():
    print("10. hasil_uji_signifikansi_paired_test.csv")
if "bootstrap_research_score_df" in globals():
    print("11. hasil_uji_bootstrap_research_score.csv")


In [ ]:
audit_data_df.to_csv("audit_data_details.csv", index=False)

## 24. Replikasi dan audit



In [ ]:
# 24A. Parameter protokol dalam laporan (untuk audit/reproduksi)
PAPER_CONFIG = {
    'records':3229, 'train':2583, 'test':646, 'train_after_mlsmote':6883,
    'bert_dimensions':768, 'pearson_top_k':50,
    'swarm_size':10, 'max_iterations':15, 'inertia':0.70,
    'c1':1.50, 'c2':1.50, 'patience':5,
    'rf_final_trees':150, 'rf_final_depth':14,
    'rf_fitness_trees':70, 'rf_fitness_depth':10,
    'alpha':0.35, 'beta':0.25, 'gamma':0.25,
    'delta':0.05, 'epsilon':0.05, 'zeta':0.05
}
print('Konfigurasi yang dilaporkan:')
display(pd.DataFrame(PAPER_CONFIG.items(), columns=['Parameter','Nilai laporan']))
print('Populasi notebook:', PSO_POP_SIZE, '| populasi laporan:', PAPER_CONFIG['swarm_size'])
print('Kandidat default notebook:', DEFAULT_CANDIDATE_TOP_K, '| kandidat laporan:', PAPER_CONFIG['pearson_top_k'])


In [ ]:
# 24B. Tabel 2: audit distribusi positif per label sebelum resampling
paper_labels = ['SU','DPAL','DSTI','DA','DPMBKA','DK','Library']
paper_train = [460,697,975,126,35,33,199]
paper_test = [103,198,230,29,4,10,48]
paper_distribution = pd.DataFrame({'Label':paper_labels,'Minority':['No','No','No','Yes','Yes','Yes','Yes'], 'Train positives (paper)':paper_train,'Test positives (paper)':paper_test})
paper_distribution['Total positives (paper)'] = paper_distribution['Train positives (paper)'] + paper_distribution['Test positives (paper)']
if 'Y_train' in globals() and 'Y_test' in globals():
    paper_distribution['Train positives (actual)'] = [int(np.asarray(Y_train)[:,list(LABELS).index(l)].sum()) for l in paper_labels]
    paper_distribution['Test positives (actual)'] = [int(np.asarray(Y_test)[:,list(LABELS).index(l)].sum()) for l in paper_labels]
    paper_distribution['Train match'] = paper_distribution['Train positives (actual)'].eq(paper_distribution['Train positives (paper)'])
    paper_distribution['Test match'] = paper_distribution['Test positives (actual)'].eq(paper_distribution['Test positives (paper)'])
display(paper_distribution)


In [ ]:
# 24C. Tabel 4 dan 5: seluruh metrik utama dan trade-off
paper_overall = pd.DataFrame([
 ['All Features',768,1.0000,.7751,.5511,.2950,.0537,.4034],
 ['Pearson Top-50',50,.0651,.7722,.5543,.3088,.0564,.4540],
 ['Standard PSO',400,.5208,.7736,.5463,.2898,.0542,.4241],
 ['IA-MPSO',22,.0286,.7590,.5510,.3307,.0599,.4573]],
 columns=['Method','Features','Feature Ratio','Weighted F1','Macro-F1','Minority Recall','Hamming Loss','Research Score'])
display(paper_overall)
if 'comparison_results_df' in globals():
    actual = comparison_results_df.copy()
    cols = {'Selected Features':'Features','F1 Weighted':'Weighted F1','F1 Macro':'Macro-F1'}
    actual = actual.rename(columns=cols)
    shared = [c for c in paper_overall.columns if c in actual.columns and c != 'Method']
    audit = paper_overall.merge(actual[['Method']+shared],on='Method',how='left',suffixes=(' (paper)',' (actual)'))
    for c in shared:
        audit[c+' (difference)'] = pd.to_numeric(audit[c+' (actual)'],errors='coerce') - audit[c+' (paper)']
    display(audit)
    audit.to_csv('audit_comparison_paper_vs_actual.csv',index=False)


In [ ]:
# 24D. Tabel 6 dan Gambar 3: per-label recall yang dilaporkan
import matplotlib.pyplot as plt
paper_recall = pd.DataFrame({'Label':paper_labels,
 'Minority':['No','No','No','Yes','Yes','Yes','Yes'],
 'Standard PSO (paper)':[.5300,.6900,.8600,.2800,.1800,.1500,.5492],
 'IA-MPSO (paper)':[.5534,.7071,.8739,.3103,.2500,.2000,.5625]})
paper_recall['Delta (paper)'] = paper_recall['IA-MPSO (paper)']-paper_recall['Standard PSO (paper)']
paper_recall['Relative gain (%)'] = 100*paper_recall['Delta (paper)']/paper_recall['Standard PSO (paper)']
display(paper_recall)
xx=np.arange(len(paper_labels)); fig,ax=plt.subplots(figsize=(11,4))
ax.bar(xx-.2,paper_recall['Standard PSO (paper)'],width=.4,label='Standard PSO')
ax.bar(xx+.2,paper_recall['IA-MPSO (paper)'],width=.4,label='IA-MPSO')
ax.set_xticks(xx,paper_labels);ax.set_ylabel('Recall');ax.set_ylim(0,1);ax.legend();ax.set_title('Recall per label: angka laporan (bukan hasil eksekusi)');plt.tight_layout();plt.show()
# Audit recall aktual hanya jika prediksi masing-masing metode tersedia
if 'metaheuristic_artifacts' in globals():
    print('Kunci artifact tersedia:',list(metaheuristic_artifacts.keys()))
    print('Untuk audit recall aktual, gunakan prediksi test yang tersimpan pada kedua artifact; jangan mengganti angka referensi.')


In [ ]:
# 24E. Gambar 2: feature compactness dan minority recall
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].bar(paper_overall['Method'],paper_overall['Features']);axes[0].set_ylabel('Jumlah dimensi');axes[0].tick_params(axis='x',rotation=20)
axes[1].bar(paper_overall['Method'],paper_overall['Minority Recall']);axes[1].set_ylabel('Minority Recall');axes[1].tick_params(axis='x',rotation=20)
fig.suptitle('Gambar 2: nilai referensi artikel');plt.tight_layout();plt.show()


In [ ]:
# 24F. Tabel 7: signifikansi yang dilaporkan
paper_significance=pd.DataFrame([
 ['IA-MPSO vs All Features','Paired t-test',.0220,'Significant'],
 ['IA-MPSO vs Pearson Top-50','Paired t-test',.5030,'Not significant'],
 ['IA-MPSO vs Standard PSO','Paired t-test',.1317,'Not significant']],
 columns=['Comparison','Reported test','Reported p-value','Reported decision'])
display(paper_significance)
print('P-value di atas adalah referensi artikel, bukan p-value yang dihitung ulang.')
print('Audit inferensial memerlukan definisi unit pasangan, statistik uji, dan prediksi yang identik.')


## 25. Uji coba khusus Tabel 6: Standard PSO vs IA-MPSO per label



In [ ]:
# 25A. Replikasi tabel dan grafik Tabel 6 berdasarkan angka yang tercantum dalam laporan
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

LABEL_ORDER = ['SU', 'DPAL', 'DSTI', 'DA', 'DPMBKA', 'DK', 'Library']
MINORITY_PAPER = ['DA', 'DPMBKA', 'DK', 'Library']
standard_report = np.array([0.5300,0.6900,0.8600,0.2800,0.1800,0.1500,0.5492])
modified_report = np.array([0.5534,0.7071,0.8739,0.3103,0.2500,0.2000,0.5625])

tabel6_referensi = pd.DataFrame({
    'Label': LABEL_ORDER,
    'Minority': ['Yes' if l in MINORITY_PAPER else 'No' for l in LABEL_ORDER],
    'Standard PSO Recall': standard_report,
    'IA-MPSO Recall': modified_report,
})
tabel6_referensi['Δ Recall'] = tabel6_referensi['IA-MPSO Recall'] - tabel6_referensi['Standard PSO Recall']
tabel6_referensi['Relative gain (%)'] = np.where(standard_report != 0, 100*tabel6_referensi['Δ Recall']/standard_report, np.nan)
minority_avg = tabel6_referensi.loc[tabel6_referensi['Minority']=='Yes', ['Standard PSO Recall','IA-MPSO Recall']].mean()
summary = pd.DataFrame([{'Label':'Minority avg.','Minority':'4 labels',
                         'Standard PSO Recall': minority_avg.iloc[0],
                         'IA-MPSO Recall': minority_avg.iloc[1],
                         'Δ Recall': minority_avg.iloc[1]-minority_avg.iloc[0],
                         'Relative gain (%)':100*(minority_avg.iloc[1]-minority_avg.iloc[0])/minority_avg.iloc[0]}])
tabel6_lengkap = pd.concat([tabel6_referensi, summary],ignore_index=True)
print('TABEL 6 - ANGKA REFERENSI DARI LAPORAN (bukan hasil jalankan model)')
display(tabel6_lengkap.style.format({'Standard PSO Recall':'{:.4f}', 'IA-MPSO Recall':'{:.4f}', 'Δ Recall':'{:+.4f}', 'Relative gain (%)':'{:+.2f}%'}))

x=np.arange(len(LABEL_ORDER)); width=.35
fig,ax=plt.subplots(figsize=(12,6))
b1=ax.bar(x-width/2, standard_report,width,label='Standard PSO',color='#1473b6')
b2=ax.bar(x+width/2, modified_report,width,label='IA-MPSO',color='#ff801b')
for bars in (b1,b2):
    for b in bars:
        ax.annotate(f'{b.get_height():.3f}',(b.get_x()+b.get_width()/2,b.get_height()),
                    xytext=(0,4),textcoords='offset points',ha='center',fontsize=8)
ax.set_xticks(x,LABEL_ORDER);ax.set_ylabel('Recall');ax.set_ylim(0,1.0)
ax.set_title('Gambar 3. Per-label Recall: Standard PSO vs IA-MPSO (referensi laporan)')
ax.legend();ax.grid(axis='y',alpha=.20)
plt.tight_layout();plt.show()
tabel6_lengkap.to_csv('tabel6_referensi_laporan.csv',index=False)


In [ ]:
# 25B. UJI COBA AKTUAL: hitung recall dari prediksi kedua model pada Y_test yang sama.
# Harus menjalankan seluruh tahap pipeline terlebih dahulu, khususnya tahap 16.
from sklearn.metrics import recall_score, confusion_matrix
required = ['metaheuristic_artifacts','Y_test','LABELS']
missing = [v for v in required if v not in globals()]
if missing:
    print('Belum dapat menghitung hasil aktual. Jalankan sel pipeline sebelumnya:', missing)
else:
    artifacts = metaheuristic_artifacts
    keys = ['Standard PSO','Imbalance-Aware Modified PSO']
    if any(k not in artifacts for k in keys):
        print('Artifact prediksi belum lengkap. Diperlukan:', keys)
    else:
        y_true = np.asarray(Y_test)
        pred_std = np.asarray(artifacts['Standard PSO']['y_pred'])
        pred_ia = np.asarray(artifacts['Imbalance-Aware Modified PSO']['y_pred'])
        label_list=list(LABELS)
        assert y_true.shape == pred_std.shape == pred_ia.shape, 'Dimensi Y_test dan prediksi tidak sama!'
        assert set(LABEL_ORDER)==set(label_list), 'Nama label tidak cocok dengan Tabel 6.'
        order=[label_list.index(l) for l in LABEL_ORDER]
        y_true=y_true[:,order]; pred_std=pred_std[:,order]; pred_ia=pred_ia[:,order]
        actual_std = recall_score(y_true,pred_std,average=None,zero_division=0)
        actual_ia = recall_score(y_true,pred_ia,average=None,zero_division=0)
        tp_std=np.sum((y_true==1)&(pred_std==1),axis=0)
        tp_ia=np.sum((y_true==1)&(pred_ia==1),axis=0)
        positives=np.sum(y_true==1,axis=0)
        tabel6_aktual=pd.DataFrame({
            'Label':LABEL_ORDER,'Minority':['Yes' if l in MINORITY_PAPER else 'No' for l in LABEL_ORDER],
            'Test positives':positives,'TP Standard PSO':tp_std,'TP IA-MPSO':tp_ia,
            'Standard PSO Recall (actual)':actual_std,'IA-MPSO Recall (actual)':actual_ia,
            'Standard PSO Recall (paper)':standard_report,'IA-MPSO Recall (paper)':modified_report
        })
        tabel6_aktual['Δ Recall (actual)']=actual_ia-actual_std
        tabel6_aktual['Relative gain (%) (actual)']=np.where(actual_std>0,100*(actual_ia-actual_std)/actual_std,np.nan)
        tabel6_aktual['Error Std vs Paper']=actual_std-standard_report
        tabel6_aktual['Error IA vs Paper']=actual_ia-modified_report
        print('HASIL PENGUJIAN AKTUAL (bukan angka yang disalin dari laporan)')
        display(tabel6_aktual.round(4))
        sel=np.array([l in MINORITY_PAPER for l in LABEL_ORDER])
        print('Minority Recall aktual Standard PSO =',round(float(actual_std[sel].mean()),4))
        print('Minority Recall aktual IA-MPSO =',round(float(actual_ia[sel].mean()),4))
        print('Majority Recall aktual Standard PSO =',round(float(actual_std[~sel].mean()),4))
        print('Majority Recall aktual IA-MPSO =',round(float(actual_ia[~sel].mean()),4))
        print('Perbedaan terbesar absolut IA terhadap laporan =',round(float(np.max(np.abs(actual_ia-modified_report))),4))
        print('Perbedaan terbesar absolut Standard terhadap laporan =',round(float(np.max(np.abs(actual_std-standard_report))),4))
        tabel6_aktual.to_csv('tabel6_hasil_aktual_vs_laporan.csv',index=False)
        fig,ax=plt.subplots(figsize=(12,5))
        bs=ax.bar(x-width/2,actual_std,width,label='Standard PSO - aktual')
        bi=ax.bar(x+width/2,actual_ia,width,label='IA-MPSO - aktual')
        for bars in (bs,bi):
            for b in bars:
                ax.annotate(f'{b.get_height():.3f}',(b.get_x()+b.get_width()/2,b.get_height()),xytext=(0,3),textcoords='offset points',ha='center',fontsize=8)
        ax.set_xticks(x,LABEL_ORDER); ax.set_ylabel('Recall');ax.set_ylim(0,1)
        ax.set_title('Hasil uji coba aktual pada data test: Standard PSO vs IA-MPSO')
        ax.legend();ax.grid(axis='y',alpha=.2);plt.tight_layout();plt.show()
        fig.savefig('grafik_recall_per_label_hasil_aktual.png',dpi=200,bbox_inches='tight')
        print('Ekspor: tabel6_hasil_aktual_vs_laporan.csv dan grafik_recall_per_label_hasil_aktual.png')
